# 14 · Ensembles — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch14_ensembles/03_notebook.ipynb)

La partie 0 vérifie tes réponses courtes aux quiz, aux rappels et aux exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wb.check`, ou les tests de ta librairie `mylearn`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Les questions « dans tes notes » qui n'ont pas de cellule 📝 se notent dans la section « Notes sur le notebook » de ta copie de `06_mes_reponses.md`. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/ch14_ensembles/03_notebook.ipynb`, créée par `python tools/start_chapter.py 14`) : ce fichier-ci est mis à jour par Claude. Sur Colab, le badge ouvre cette version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 14.Q1 à 14.Q11, 14.R1 à 14.R3, 14.1 à 14.3, 14.5 à 14.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ ∂ | ★ à ★★★ | |
| A | 14.11 | Trois classifieurs du ch. 13 votent : mieux que le meilleur ? | 🔮 | ★ | 10 |
|  | 14.12 | Coder le vote pondéré et le tirage bootstrap | 🔨 | ★★ | 25 |
|  | 14.13 | Jury de votants : indépendants puis corrélés | 🔬 | ★★ | 30 |
|  | 14.14 | Voting et stacking avec les classifieurs du ch. 13 | 📦 | ★★ | 25 |
| B | 14.15 | Forêts scikit-learn sur California : OOB et parallélisme | 📦 | ★★ | 25 |
|  | 14.16 | Combien d'arbres ? Bagging, forêt et ExtraTrees face au bruit | 🔬 | ★★ | 30 |
|  | 14.17 | Coder BaggingClassifier avec score out-of-bag | 🔨 | ★★★ | 50 |
|  | 14.18 | Coder RandomForest et ExtraTrees | 🔨 | ★★★ | 45 |
|  | 14.19 | Importance des features : impureté contre permutation | 📦 | ★★★ | 35 |
| C | 14.20 | Lire les courbes train/validation d'un boosting | 📈 | ★ | 15 |
|  | 14.21 | Learning rate et nombre d'arbres : prédire les courbes | 🔮 | ★ | 15 |
|  | 14.22 | Frontières d'AdaBoost après 1, 3, 10 et 50 souches | 🎨 | ★★ | 25 |
|  | 14.23 | L'AdaBoost qui n'apprend rien | 🐛 | ★★ | 25 |
|  | 14.24 | Coder AdaBoost (SAMME) avec des souches pondérées | 🔨 | ★★★ | 75 |
|  | 14.25 | Coder un gradient boosting de régression | 🔨 | ★★★ | 60 |
| D | 14.26 | HistGradientBoosting, XGBoost et LightGBM sur California | 📦 | ★★★ | 45 |
|  | 14.27 | Un benchmark reproductible : graines, versions et tableau de résultats | 🛠️ | ★★ | 25 |
|  | 14.28 | Défi : RMSE ≤ 0,45 sur California | 🏆 | ★★★★ | 100 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")  # your own library

## Objectifs et rappel express

- Calculer à la main un vote pondéré, la probabilité qu'une majorité se trompe, la part des exemples oubliés par un bootstrap, une itération d'AdaBoost et deux itérations de gradient boosting.
- Écrire et vérifier contre scikit-learn les ensembles de ta librairie, construits sur tes arbres du ch. 13 : `plurality_vote`, `bootstrap_indices`, `BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier` et `GradientBoostingRegressor`.
- Utiliser les forêts de scikit-learn, HistGradientBoosting, XGBoost et LightGBM avec early stopping, et interpréter l'importance des features.
- Choisir entre bagging et boosting, et rendre un benchmark reproductible.

**Rappel express.** Vote pondéré : $\hat y = \arg\max_k \sum_m w_m\,\mathbb{1}[h_m(\mathbf{x}) = k]$ ; majorité de $M$ votants indépendants, chacun juste avec la probabilité $q$ : $\sum_{j > M/2} \binom{M}{j} q^j (1 - q)^{M - j}$ ; jamais tiré par un bootstrap : $(1 - 1/n)^n \to e^{-1} \approx 0{,}368$ ; variance d'une moyenne corrélée : $\rho\sigma^2 + (1 - \rho)\sigma^2/M$ ; AdaBoost (SAMME) : $\varepsilon_m = \sum_i w_i \mathbb{1}[h_m(\mathbf{x}_i) \ne y_i] / \sum_i w_i$, $\alpha_m = \eta\big(\ln\frac{1 - \varepsilon_m}{\varepsilon_m} + \ln(K - 1)\big)$, $w_i \leftarrow w_i e^{\alpha_m}$ pour les mal classés ; gradient boosting : $F_0 = \bar y$, $r_i = y_i - F_{m-1}(\mathbf{x}_i)$, $F_m = F_{m-1} + \eta\, h_m$. En scikit-learn : `sklearn.ensemble` (`BaggingClassifier`, `RandomForestClassifier`, `ExtraTreesClassifier`, `AdaBoostClassifier`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, `VotingClassifier`, `StackingClassifier`), `sklearn.inspection.permutation_importance` ; `xgboost.XGBRegressor`, `lightgbm.LGBMRegressor`.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 14.1, 14.2, 14.5 à 14.7 et ∂ 14.3)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"E"`) ; une suite de codes : les lettres dans l'ordre (`"XYZ"`) ; plusieurs nombres : une liste (`[4, 7]`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la preuve ∂ 14.4, la réflexion (🗣️ 14.8, ⚖️ 14.9, 📄 14.10) et l'entretien se corrigent avec `05_solutions.md`.

**14.Q1 — Pourquoi un groupe peut battre chacun de ses membres**

In [ ]:
# 14.Q1: the values you found on paper
answer_14_Q1a = ...  # a) True or False
answer_14_Q1b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q1c = ...  # c) True or False
answer_14_Q1d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q1a, answer_14_Q1b, answer_14_Q1c, answer_14_Q1d]):
    wb.check(f"14.Q1{letter}", answer)

**14.Q2 — Un ensemble corrige-t-il un biais des données ?**

In [ ]:
# 14.Q2: the values you found on paper
answer_14_Q2a = ...  # a) True or False
answer_14_Q2b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q2c = ...  # c) True or False
answer_14_Q2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q2a, answer_14_Q2b, answer_14_Q2c, answer_14_Q2d]):
    wb.check(f"14.Q2{letter}", answer)

**14.Q3 — Pluralité, majorité et vote pondéré**

In [ ]:
# 14.Q3: the values you found on paper
answer_14_Q3a = ...  # a) a class: "A", "B" or "C"
answer_14_Q3b = ...  # b) True or False
answer_14_Q3c = ...  # c) a number
answer_14_Q3d = ...  # d) a class: "A", "B" or "C"

for letter, answer in zip("abcd", [answer_14_Q3a, answer_14_Q3b, answer_14_Q3c, answer_14_Q3d]):
    wb.check(f"14.Q3{letter}", answer)

**14.Q4 — Tirer avec remise**

In [ ]:
# 14.Q4: the values you found on paper
answer_14_Q4a = ...  # a) True or False
answer_14_Q4b = ...  # b) a whole number
answer_14_Q4c = ...  # c) a whole number
answer_14_Q4d = ...  # d) True or False
answer_14_Q4e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q4a, answer_14_Q4b, answer_14_Q4c, answer_14_Q4d, answer_14_Q4e]):
    wb.check(f"14.Q4{letter}", answer)

**14.Q5 — Ce que le bagging réduit**

In [ ]:
# 14.Q5: the values you found on paper
answer_14_Q5a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q5b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q5c = ...  # c) True or False
answer_14_Q5d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_Q5a, answer_14_Q5b, answer_14_Q5c, answer_14_Q5d]):
    wb.check(f"14.Q5{letter}", answer)

**14.Q6 — Le hasard d'une forêt aléatoire**

In [ ]:
# 14.Q6: the values you found on paper
answer_14_Q6a = ...  # a) a whole number
answer_14_Q6b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q6c = ...  # c) True or False
answer_14_Q6d = ...  # d) two letters, e.g. "EF"
answer_14_Q6e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q6a, answer_14_Q6b, answer_14_Q6c, answer_14_Q6d, answer_14_Q6e]):
    wb.check(f"14.Q6{letter}", answer)

**14.Q7 — ExtraTrees : encore plus de hasard**

In [ ]:
# 14.Q7: the values you found on paper
answer_14_Q7a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q7b = ...  # b) True or False
answer_14_Q7c = ...  # c) True or False
answer_14_Q7d = ...  # d) the letter of your choice, e.g. "E"

for letter, answer in zip("abcd", [answer_14_Q7a, answer_14_Q7b, answer_14_Q7c, answer_14_Q7d]):
    wb.check(f"14.Q7{letter}", answer)

**14.Q8 — Apprenant faible, souche et apprenant fort**

In [ ]:
# 14.Q8: the values you found on paper
answer_14_Q8a = ...  # a) a whole number
answer_14_Q8b = ...  # b) a whole number
answer_14_Q8c = ...  # c) True or False
answer_14_Q8d = ...  # d) the letter of your choice, e.g. "E"
answer_14_Q8e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q8a, answer_14_Q8b, answer_14_Q8c, answer_14_Q8d, answer_14_Q8e]):
    wb.check(f"14.Q8{letter}", answer)

**14.Q9 — Ce que repondère AdaBoost**

In [ ]:
# 14.Q9: the values you found on paper
answer_14_Q9a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q9b = ...  # b) a number
answer_14_Q9c = ...  # c) the letter of your choice, e.g. "E"
answer_14_Q9d = ...  # d) True or False
answer_14_Q9e = ...  # e) a number

for letter, answer in zip("abcde", [answer_14_Q9a, answer_14_Q9b, answer_14_Q9c, answer_14_Q9d, answer_14_Q9e]):
    wb.check(f"14.Q9{letter}", answer)

**14.Q10 — Bagging ou boosting ?**

In [ ]:
# 14.Q10: the values you found on paper
answer_14_Q10a = ...  # a) six letters B or O, e.g. "BBBBBB"
answer_14_Q10b = ...  # b) the letter of your choice, e.g. "E"
answer_14_Q10c = ...  # c) True or False

for letter, answer in zip("abc", [answer_14_Q10a, answer_14_Q10b, answer_14_Q10c]):
    wb.check(f"14.Q10{letter}", answer)

**14.Q11 — Gradient boosting : corriger les résidus**

In [ ]:
# 14.Q11: the values you found on paper
answer_14_Q11a = ...  # a) the letter of your choice, e.g. "E"
answer_14_Q11b = ...  # b) a number
answer_14_Q11c = ...  # c) a number
answer_14_Q11d = ...  # d) True or False
answer_14_Q11e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_Q11a, answer_14_Q11b, answer_14_Q11c, answer_14_Q11d, answer_14_Q11e]):
    wb.check(f"14.Q11{letter}", answer)

**14.R1 — L'impureté de Gini d'un nœud**

In [ ]:
# 14.R1: the values you found on paper
answer_14_R1a = ...  # a) a number
answer_14_R1b = ...  # b) a number with 3 decimals
answer_14_R1c = ...  # c) a number
answer_14_R1d = ...  # d) a number with 3 decimals
answer_14_R1e = ...  # e) True or False

for letter, answer in zip("abcde", [answer_14_R1a, answer_14_R1b, answer_14_R1c, answer_14_R1d, answer_14_R1e]):
    wb.check(f"14.R1{letter}", answer)

**14.R2 — Induction et généralisation**

In [ ]:
# 14.R2: the values you found on paper
answer_14_R2a = ...  # a) the letter of your choice, e.g. "E"
answer_14_R2b = ...  # b) True or False
answer_14_R2c = ...  # c) the letter of your choice, e.g. "E"
answer_14_R2d = ...  # d) True or False

for letter, answer in zip("abcd", [answer_14_R2a, answer_14_R2b, answer_14_R2c, answer_14_R2d]):
    wb.check(f"14.R2{letter}", answer)

**14.R3 — Un-contre-tous et un-contre-un**

In [ ]:
# 14.R3: the values you found on paper
answer_14_R3a = ...  # a) a whole number
answer_14_R3b = ...  # b) a whole number
answer_14_R3c = ...  # c) a whole number
answer_14_R3d = ...  # d) True or False
answer_14_R3e = ...  # e) a number

for letter, answer in zip("abcde", [answer_14_R3a, answer_14_R3b, answer_14_R3c, answer_14_R3d, answer_14_R3e]):
    wb.check(f"14.R3{letter}", answer)

**Ex 14.1 — Dépouiller un vote pondéré**

In [ ]:
# Ex 14.1: the values you found on paper
answer_14_1a = ...  # a) a number
answer_14_1b = ...  # b) a class: 0, 1 or 2
answer_14_1c = ...  # c) a class: 0, 1 or 2
answer_14_1d = ...  # d) a list of four classes
answer_14_1e = ...  # e) a list of four classes
answer_14_1f = ...  # f) a whole number
answer_14_1g = ...  # g) True or False
answer_14_1h = ...  # h) True or False

for letter, answer in zip("abcdefgh", [answer_14_1a, answer_14_1b, answer_14_1c, answer_14_1d, answer_14_1e, answer_14_1f, answer_14_1g, answer_14_1h]):
    wb.check(f"14.1{letter}", answer)

**Ex 14.2 — Additionner les voix de trois droites, région par région**

In [ ]:
# Ex 14.2: the values you found on paper
answer_14_2a = ...  # a) a number
answer_14_2b = ...  # b) a number
answer_14_2c = ...  # c) a class: 0 or 1
answer_14_2d = ...  # d) a whole number
answer_14_2e = ...  # e) a list of five classes
answer_14_2f = ...  # f) a whole number
answer_14_2g = ...  # g) a number
answer_14_2h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_14_2a, answer_14_2b, answer_14_2c, answer_14_2d, answer_14_2e, answer_14_2f, answer_14_2g, answer_14_2h]):
    wb.check(f"14.2{letter}", answer)

**Ex 14.3 — Probabilité d'être oublié par un bootstrap : vers 36,8 %**

In [ ]:
# Ex 14.3: the values you found on paper
answer_14_3a = ...  # a) a number with 3 decimals
answer_14_3b = ...  # b) a number with 2 decimals
answer_14_3c = ...  # c) a number with 4 decimals
answer_14_3d = ...  # d) a number with 4 decimals
answer_14_3e = ...  # e) a number with 4 decimals
answer_14_3f = ...  # f) a number with 4 decimals
answer_14_3g = ...  # g) a number with 1 decimal
answer_14_3h = ...  # h) a number with 3 decimals

for letter, answer in zip("abcdefgh", [answer_14_3a, answer_14_3b, answer_14_3c, answer_14_3d, answer_14_3e, answer_14_3f, answer_14_3g, answer_14_3h]):
    wb.check(f"14.3{letter}", answer)

**Ex 14.5 — Quand la majorité de votants indépendants se trompe**

In [ ]:
# Ex 14.5: the values you found on paper
answer_14_5a = ...  # a) a number with 3 decimals
answer_14_5b = ...  # b) a number with 3 decimals
answer_14_5c = ...  # c) a number with 3 decimals
answer_14_5d = ...  # d) a number with 4 decimals
answer_14_5e = ...  # e) a number with 4 decimals
answer_14_5f = ...  # f) a number
answer_14_5g = ...  # g) an odd whole number
answer_14_5h = ...  # h) a number with 3 decimals
answer_14_5i = ...  # i) a number with 3 decimals

for letter, answer in zip("abcdefghi", [answer_14_5a, answer_14_5b, answer_14_5c, answer_14_5d, answer_14_5e, answer_14_5f, answer_14_5g, answer_14_5h, answer_14_5i]):
    wb.check(f"14.5{letter}", answer)

**Ex 14.6 — Une itération d'AdaBoost à la main**

In [ ]:
# Ex 14.6: the values you found on paper
answer_14_6a = ...  # a) a number
answer_14_6b = ...  # b) a number with 3 decimals
answer_14_6c = ...  # c) a number
answer_14_6d = ...  # d) a number with 4 decimals
answer_14_6e = ...  # e) a list [threshold, class on the left]
answer_14_6f = ...  # f) a number with 4 decimals
answer_14_6g = ...  # g) a number with 3 decimals
answer_14_6h = ...  # h) a list of eight classes
answer_14_6i = ...  # i) a number with 4 decimals
answer_14_6j = ...  # j) a number with 3 decimals
answer_14_6k = ...  # k) a list of eight classes
answer_14_6l = ...  # l) a number with 3 decimals
answer_14_6m = ...  # m) True or False

for letter, answer in zip("abcdefghijklm", [answer_14_6a, answer_14_6b, answer_14_6c, answer_14_6d, answer_14_6e, answer_14_6f, answer_14_6g, answer_14_6h, answer_14_6i, answer_14_6j, answer_14_6k, answer_14_6l, answer_14_6m]):
    wb.check(f"14.6{letter}", answer)

**Ex 14.7 — Deux itérations de gradient boosting sur cinq points**

In [ ]:
# Ex 14.7: the values you found on paper
answer_14_7a = ...  # a) a number
answer_14_7b = ...  # b) a list of five numbers
answer_14_7c = ...  # c) a threshold
answer_14_7d = ...  # d) a list [left value, right value], 3 decimals
answer_14_7e = ...  # e) a number with 2 decimals
answer_14_7f = ...  # f) a number with 3 decimals
answer_14_7g = ...  # g) a threshold
answer_14_7h = ...  # h) a number with 4 decimals
answer_14_7i = ...  # i) a number with 4 decimals
answer_14_7j = ...  # j) a number with 4 decimals
answer_14_7k = ...  # k) a number

for letter, answer in zip("abcdefghijk", [answer_14_7a, answer_14_7b, answer_14_7c, answer_14_7d, answer_14_7e, answer_14_7f, answer_14_7g, answer_14_7h, answer_14_7i, answer_14_7j, answer_14_7k]):
    wb.check(f"14.7{letter}", answer)

## Partie A · Faire voter des modèles

Trois classifieurs du ch. 13 qui votent (14.11), ton vote pondéré et ton bootstrap (14.12), ce que la corrélation fait à un jury (14.13), puis voting et stacking avec scikit-learn (14.14). La cellule suivante importe les ensembles de scikit-learn, définit les outils des parties A à D et découpe les 333 manchots complets (233 pour l'entraînement, 100 pour le test), comme au ch. 13. Ne recopie jamais un nombre à la main : chaque réponse est calculée par ton code, et pas arrondie.

In [ ]:
# Tools for the notebook exercises (parts A to D), and the penguins
import contextlib
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, ExtraTreesClassifier, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor,
                              StackingClassifier, VotingClassifier)
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from wb.errors import MylearnMissingError


def verdict(ex_id, ok, success, failure):
    """Print ✅ with the success message, or ❌ with the failure message."""
    print(f"✅ Ex {ex_id} : {success}" if ok else f"❌ Ex {ex_id} : {failure}")


def filled(*values):
    """True when none of the values is still `...` (or None): the answer has been written."""
    return all(value is not ... and value is not None for value in values)


def fr(x, digits=3):
    """A number written the French way, with a decimal comma (for the messages in French)."""
    return f"{x:.{digits}f}".replace(".", ",")


def returned(ex_id, name, value):
    """False, with a ❌ message, when the function `name` returned None (a forgotten return); True otherwise."""
    if value is None:
        print(f"❌ Ex {ex_id} : {name} renvoie None : as-tu oublié le return ?")
        return False
    return True


def print_answer(ex_id, value, **_):
    """Solutions notebook: show the value that the exercise notebook checks with wb.check."""
    try:
        shown = np.round(np.asarray(value, dtype=float), 4).tolist() if isinstance(value, (np.ndarray, list, tuple)) else value
    except (TypeError, ValueError):     # a list of names
        shown = list(value)
    print(f"{ex_id}:", shown)


def fitted(model, X, y=None, **fit_params):
    """The model after model.fit(X, y): no chained call, so that a fit that forgets `return self` (the tests report
    it) does not stop the check cells."""
    if y is None:
        model.fit(X, **fit_params)
    else:
        model.fit(X, y, **fit_params)
    return model


@contextlib.contextmanager
def checked(ex_id):
    """Like wb.attempt (⏳ while the code is not written yet), but any other error raised by YOUR code becomes one ❌
    line instead of stopping the cell, so that the tests run after it can say more."""
    with wb.attempt(ex_id):
        try:
            yield
        except (NotImplementedError, ImportError, MylearnMissingError):
            raise
        except Exception as exc:                              # a bug in the learner's code: report it, go on
            if isinstance(exc, TypeError) and "NoneType" in str(exc):
                raise                                         # wb.attempt explains the forgotten return
            hint = " ; une fonction ou une méthode oublie-t-elle son return ?" if "NoneType" in str(exc) else ""
            print(f"❌ Ex {ex_id} : ton code lève {type(exc).__name__} : {str(exc)[:300]}{hint}")


def written(probe):
    """False while probe() stops on code not written yet (NotImplementedError, or no mylearn at all); True otherwise,
    even when it raises another error (the tests then say which)."""
    try:
        probe()
    except (NotImplementedError, ImportError, MylearnMissingError):
        return False
    except Exception:
        return True
    return True


def rmse(y_true, y_pred):
    """Root mean squared error, a Python float."""
    return float(np.sqrt(np.mean((np.asarray(y_true, dtype=float) - np.asarray(y_pred, dtype=float)) ** 2)))


TEST_FILES = {"ensemble": "tests/test_ch14_ensemble.py", "tree": "tests/test_ch13_tree.py"}


def run_mylearn_tests(module, keyword, impl="learner"):
    """Run the tests of mylearn.<module> selected by `keyword` (on YOUR code by default)."""
    test_file = TEST_FILES[module]
    command = [sys.executable, "-m", "pytest", test_file, "-k", keyword, "-q", "-p", "no:cacheprovider",
               "--color=no", "-rf", "--tb=no"]
    if impl != "learner":
        command.append(f"--impl={impl}")
    try:
        result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, timeout=600,
                                env={**os.environ, "COLUMNS": "1000"})   # long lines: the reason of each failure
    except subprocess.TimeoutExpired:
        print("❌ les tests n'ont pas fini en 10 min : une boucle de ton code ne s'arrête-t-elle pas ?")
        return
    lines = result.stdout.strip().splitlines()
    failed = [line for line in lines if line.startswith("FAILED ")]
    if any("'NoneType' object has no attribute" in line for line in failed):
        print("💡 un attribut est lu sur None : une méthode oublie-t-elle son return ?")
    for line in failed[:8]:                                  # the test, then the reason of its failure
        name, _, reason = line.removeprefix(f"FAILED {test_file}::").partition(" - ")
        print(f"❌ {name}\n   {reason[:800]}")
    if len(failed) > 8:
        print(f"   ... and {len(failed) - 8} other failed test(s)")
    print("pytest:", lines[-1] if lines else result.stderr.strip()[-300:])


PENGUINS = wb.datasets.load_penguins(dropna=True).reset_index(drop=True)   # the 333 complete penguins
MEASURES = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
P_TRAIN, P_TEST = train_test_split(PENGUINS, test_size=100, random_state=1314, stratify=PENGUINS["species"])
print(len(P_TRAIN), "training penguins,", len(P_TEST), "test penguins")

### Ex 14.11 — Trois classifieurs du ch. 13 votent : mieux que le meilleur ? 🔮 ★ ⏱️ 10 min
**Objectif :** prévoir, avant de le mesurer, ce qu'un vote apporte à trois classifieurs entraînés sur les mêmes données.  
**Prérequis :** ch. 13 (kNN, Naïve Bayes, arbres) · fiche §14.1 à §14.3 · **Fil rouge :** Penguins · **Parcours :** R, C

Les 333 manchots, décrits par **deux** mesures seulement : l'épaisseur du bec (`bill_depth_mm`) et la longueur de la nageoire (`flipper_length_mm`). Trois classifieurs du ch. 13 : A un kNN à 15 voisins sur les mesures standardisées, B `GaussianNB()`, C un arbre de profondeur 3. Chaque manchot est prédit par des modèles qui ne l'ont pas vu (validation croisée à 10 folds, `cross_val_predict`), puis les trois prédictions votent à la pluralité. Les trois membres font entre 79 et 82 % d'accuracy.

Prédis, **avant** d'exécuter quoi que ce soit (un point = 0,01 d'accuracy) :
a) `vote_11` : l'accuracy du vote, comparée à celle du meilleur des trois : `"A"` plus d'un point **sous** le meilleur ; `"B"` à moins d'un point du meilleur (au-dessus ou au-dessous) ; `"C"` de 1 à 3 points au-dessus ; `"D"` plus de 3 points au-dessus ;
b) `shared_11` : parmi les manchots que le meilleur membre classe mal, la part que les **deux autres** classent mal aussi : `"A"` moins de 25 % ; `"B"` de 25 à 50 % ; `"C"` de 50 à 75 % ; `"D"` plus de 75 % ;
c) `pair_11` : les erreurs du vote confondent surtout : `"A"` Adélie et Chinstrap ; `"B"` Adélie et Gentoo ; `"C"` Chinstrap et Gentoo ; `"D"` les erreurs se répartissent sur les trois paires.

Puis exécute l'expérience : elle affiche l'accuracy de chaque membre et du vote, les erreurs partagées, la table des espèces et des votes, et entoure les manchots mal classés.

In [ ]:
X_11 = PENGUINS[["bill_depth_mm", "flipper_length_mm"]].to_numpy()   # two measures only
Y_11 = PENGUINS["species"].to_numpy()
CV_11 = StratifiedKFold(n_splits=10, shuffle=True, random_state=1411)
MEMBERS_11 = {"A": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)),
              "B": GaussianNB(),
              "C": DecisionTreeClassifier(max_depth=3, random_state=0)}

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
vote_11 = ...     # a) "A", "B", "C" or "D"
shared_11 = ...   # b) "A", "B", "C" or "D"
pair_11 = ...     # c) "A", "B", "C" or "D"

In [ ]:
if any(answer is ... or answer is None for answer in [vote_11, shared_11, pair_11]):
    print("⏳ Ex 14.11 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    # every penguin predicted by models that did not see it (10-fold cross-validation)
    predictions_11 = {letter: cross_val_predict(model, X_11, Y_11, cv=CV_11) for letter, model in MEMBERS_11.items()}
    accuracies_11 = {letter: float(np.mean(predicted == Y_11)) for letter, predicted in predictions_11.items()}
    stacked_11 = np.array(list(predictions_11.values()))               # (3 voters, 333 penguins)
    species_11 = np.unique(Y_11)
    votes_11 = np.array([(stacked_11 == s).sum(axis=0) for s in species_11])
    vote_result_11 = species_11[np.argmax(votes_11, axis=0)]           # plurality vote, ties: the first species
    for letter, accuracy in accuracies_11.items():
        print(f"member {letter}: accuracy {accuracy:.3f}")
    print(f"plurality vote of the three: accuracy {np.mean(vote_result_11 == Y_11):.3f}")
    wrong_11 = stacked_11 != Y_11
    best_11 = int(np.argmax(list(accuracies_11.values())))
    print(f"the best member ({'ABC'[best_11]}) misclassifies {wrong_11[best_11].sum()} penguins; "
          f"the other two misclassify {wrong_11.all(axis=0)[wrong_11[best_11]].sum()} of them as well")
    print(pd.crosstab(pd.Series(Y_11, name="species"), pd.Series(vote_result_11, name="vote")))
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for s in species_11:
        ax.scatter(*X_11[Y_11 == s].T, s=14, label=s)
    ax.scatter(*X_11[vote_result_11 != Y_11].T, s=60, facecolors="none", edgecolors="black", label="vote wrong")
    ax.set_xlabel("bill_depth_mm")
    ax.set_ylabel("flipper_length_mm")
    ax.legend()
    plt.show()
    wb.check("14.11a", vote_11)
    wb.check("14.11b", shared_11)
    wb.check("14.11c", pair_11)

Dans tes notes : compare avec tes prédictions. Les deux conditions de la fiche (§14.2) sont-elles remplies par ces trois membres ? Leurs erreurs se ressemblent-elles, et que faudrait-il changer pour qu'elles se ressemblent plus ou moins ? Ce que tu as observé sur ces deux mesures vaudrait-il encore avec les quatre ?

### Ex 14.12 — Coder le vote pondéré et le tirage bootstrap 🔨 ★★ ⏱️ 25 min
**Objectif :** écrire les deux briques de tous les ensembles : un vote pondéré vectorisé et un tirage avec remise.  
**Prérequis :** ✏️ 14.1 · ∂ 14.3 · ch. 2 (bootstrap) · fiche §14.3 et §14.4 · **mylearn :** `ensemble.py` · **Parcours :** C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/ensemble.py` (créé par `python tools/start_chapter.py 14`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement, plus tes modules des chapitres précédents : `clone` (ch. 8, `model_selection.py`) et tes arbres (ch. 13, `tree.py`) ; scikit-learn est l'**oracle** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch14_ensemble.py -q` les lance tous dans un terminal (compte quelques minutes si ton arbre est lent).

Écris `plurality_vote` et `bootstrap_indices`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- `plurality_vote(predictions, weights=None)` reçoit un tableau 2-D (ou une liste de listes) de forme `(n_voters, n_samples)`, une ligne par votant, et rend un tableau `(n_samples,)` : pour chaque exemple, la classe dont la somme des poids des votants est la plus grande ; une classe que personne ne désigne a un total nul, un poids peut être négatif ou nul ; à égalité, la **plus petite** classe, dans l'ordre de tri des labels (des entiers ou des chaînes) ; `weights=None` donne une voix à chacun ; `ValueError` si `predictions` n'est pas 2-D, si `weights` n'a pas une valeur par votant ou contient `NaN` ; elle ne modifie pas ses arguments ;
- `bootstrap_indices(n_samples, n_draws=None, rng=None)` rend un tableau d'**entiers** de forme `(n_draws,)` (`n_samples` tirages si `n_draws` vaut `None`), tirés uniformément **avec remise** dans $0, \dots, n - 1$ ; `rng` accepte `None`, un entier ou un `Generator` ; deux appels avec deux générateurs de même graine rendent les mêmes indices, deux appels successifs avec le **même** générateur en rendent d'autres ; `ValueError` si `n_samples < 1` ou `n_draws < 1`.

La vérification essaie les exemples des docstrings, compare tes votes à `scipy.stats.mode` (sans poids) et à une boucle (avec des poids), mesure la part d'indices distincts d'un bootstrap, puis lance les tests.

Dans tes notes : combien d'opérations ton vote fait-il pour $M$ votants, $n$ exemples et $K$ classes ? Pourquoi `bootstrap_indices` reçoit-elle un générateur plutôt qu'une graine fixée à l'intérieur ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")   # reload your saved file
with checked("14.12"):
    doc_12 = np.array([[0, 1, 1], [0, 0, 1], [1, 0, 1]])
    plain_12 = mylearn.ensemble.plurality_vote(doc_12)
    weighted_12 = mylearn.ensemble.plurality_vote(doc_12, weights=[1.0, 1.0, 3.0])
    if returned("14.12", "plurality_vote", plain_12) and returned("14.12", "plurality_vote", weighted_12):
        print("docstring examples:", np.asarray(plain_12).tolist(), np.asarray(weighted_12).tolist(),
              "(expected [0, 0, 1] and [1, 0, 1])")
with checked("14.12"):
    ballots_12 = np.random.default_rng(1412).integers(0, 4, size=(7, 500))      # 7 voters, 500 samples, labels 0 to 3
    mine_12 = np.asarray(mylearn.ensemble.plurality_vote(ballots_12))
    oracle_12 = stats.mode(ballots_12, axis=0).mode                                # ties: the smallest label
    verdict("14.12", mine_12.shape == (500,) and np.array_equal(mine_12, oracle_12),
            "sans poids, tes 500 votes sont ceux de scipy.stats.mode.",
            ("sans poids, tes votes diffèrent de scipy.stats.mode (relis la règle d'égalité : la plus petite classe) ; "
             if mine_12.shape == (500,) else f"sans poids, ton vote rend un tableau de forme {mine_12.shape}, au lieu "
             "d'une classe par exemple (500,) ; ") + "les tests ci-dessous disent où.")
    w_12 = np.array([2.0, 1.0, 1.5, -1.0, 0.5, 0.0, 3.0])
    totals_12 = np.array([[w_12[ballots_12[:, j] == k].sum() for k in range(4)] for j in range(500)])
    verdict("14.12", np.array_equal(np.asarray(mylearn.ensemble.plurality_vote(ballots_12, w_12)),
                                    np.argmax(totals_12, axis=1)),
            "avec des poids (un négatif, un nul), tes votes sont ceux d'une boucle qui additionne les poids.",
            "avec des poids, tes votes diffèrent d'une boucle qui additionne les poids de chaque classe (un poids "
            "négatif retire des voix).")
with checked("14.12"):
    draws_12 = mylearn.ensemble.bootstrap_indices(1000, rng=np.random.default_rng(1412))
    if returned("14.12", "bootstrap_indices", draws_12):
        draws_12 = np.asarray(draws_12)
        distinct_12 = len(np.unique(draws_12)) / 1000
        print(f"bootstrap of 1000 among 1000: {distinct_12:.3f} distinct indices (about 0.632 expected)")
        verdict("14.12", draws_12.shape == (1000,) and 0.60 < distinct_12 < 0.66
                and np.array_equal(draws_12, np.asarray(mylearn.ensemble.bootstrap_indices(1000, rng=np.random.default_rng(1412))))
                and np.array_equal(np.asarray(mylearn.ensemble.bootstrap_indices(50, rng=7)),
                                   np.asarray(mylearn.ensemble.bootstrap_indices(50, rng=7))),
                "1000 tirages, environ 63 % d'indices distincts, les mêmes avec le même générateur ou le même entier.",
                "attendu : 1000 indices, environ 63 % de distincts, et les mêmes tirages pour deux générateurs de même "
                "graine (ou deux fois rng=7) ; les tests ci-dessous disent où.")
if written(lambda: mylearn.ensemble.plurality_vote(np.array([[0]]))) or written(lambda: mylearn.ensemble.bootstrap_indices(2, rng=0)):
    run_mylearn_tests("ensemble", keyword="test_plurality_vote_ or test_bootstrap_indices_")
else:
    print("⏳ Ex 14.12 : les tests attendent ton plurality_vote et ton bootstrap_indices.")

### Ex 14.13 — Jury de votants : indépendants puis corrélés 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer ce que la corrélation des erreurs coûte à un vote, et retrouver la limite par le calcul.  
**Prérequis :** ✏️ 14.5 · ch. 2 (corrélation) · fiche §14.2 (encadré sur la loi binomiale) · **Fil rouge :** synthétique · **Parcours :** R, M, C

Un jury de $M$ votants (`MS_13` : 1 à 101), chacun juste avec la probabilité $q$ = `Q_13` = 0,58, à deux classes. La fonction fournie `jury_13(M, q, c)` simule 20 000 votes : à chaque vote, avec la probabilité $c$, **tous** les votants recopient un même avis (juste avec la probabilité $q$) ; sinon, chacun décide seul. Chaque votant reste juste avec la probabilité $q$, mais leurs erreurs sont corrélées dès que $c > 0$. La majorité a raison quand plus de la moitié des votants ont raison.

Calcule :
a) `exact_13` : pour chaque $M$ de `MS_13`, la probabilité **exacte** que la majorité de $M$ votants indépendants ait raison (loi binomiale : `scipy.stats.binom`, importé sous le nom `stats`, ou `math.comb`) ;
b) `first_090_13` : le plus petit $M$ de `MS_13` dont la probabilité en a) atteint 0,9 ;
c) `simulated_13` : pour chaque $M$, la part des votes de `jury_13(M, Q_13, 0.0)` où la majorité a raison (elle doit suivre a) ;
d) `correlated_13` : la même liste avec $c = 0{,}3$ ;
e) `corr_13` : le coefficient de corrélation entre les colonnes 0 et 1 de `jury_13(11, Q_13, 0.3)` (deux votants : `np.corrcoef`) ;
f) `plateau_13` : la limite de la probabilité de d) quand $M$ grandit sans fin, **par le raisonnement** (exacte) ;
g) `q_needed_13` : avec $c = 0{,}3$, le plus petit $q$ pour lequel cette limite atteint 0,9 (exact).

La vérification contrôle a) à g), puis trace les trois courbes et ta limite.

Dans tes notes : compare e) à $c$, et explique pourquoi. Que dit f) sur un ensemble d'arbres entraînés sur les mêmes données ? Relie-le à la variance d'une moyenne corrélée (fiche §14.4).

In [ ]:
MS_13 = [1, 3, 5, 11, 21, 51, 101]   # sizes of the jury (odd)
Q_13 = 0.58                           # each voter is right with probability 0.58


def jury_13(M, q, c, n_trials=20_000, seed=1413):
    """Simulate n_trials votes of a jury of M voters, each right with probability q. In each trial, with probability c,
    every voter copies one common vote (itself right with probability q); otherwise each voter decides on its own.
    Return the boolean array of shape (n_trials, M): True where a voter is right."""
    rng = np.random.default_rng(seed)
    common = rng.random(n_trials) < q
    own = rng.random((n_trials, M)) < q
    copy = rng.random(n_trials) < c
    return np.where(copy[:, None], common[:, None], own)

In [ ]:
exact_13 = ...       # a) list: for each M of MS_13, the exact probability that the majority of M independent voters,
                     #    each right with probability Q_13, is right (binomial law)
first_090_13 = ...   # b) the smallest M of MS_13 whose probability in a) is at least 0.9
simulated_13 = ...   # c) list: for each M of MS_13, the share of the trials of jury_13(M, Q_13, 0.0) where the majority
                     #    of the voters is right
correlated_13 = ...  # d) the same list as c) with c = 0.3: jury_13(M, Q_13, 0.3)
corr_13 = ...        # e) the correlation coefficient between the columns 0 and 1 of jury_13(11, Q_13, 0.3)
plateau_13 = ...     # f) the limit of the probability in d) when M grows without bound (exact, by reasoning)
q_needed_13 = ...    # g) with c = 0.3, the smallest q for which this limit reaches 0.9 (exact)

In [ ]:
wb.check("14.13a", exact_13)
wb.check("14.13b", first_090_13)
wb.check("14.13c", simulated_13)
wb.check("14.13d", correlated_13)
wb.check("14.13e", corr_13)
wb.check("14.13f", plateau_13)
wb.check("14.13g", q_needed_13)
with wb.attempt("14.13"):
    if not filled(exact_13, simulated_13, correlated_13):
        raise NotImplementedError("exact_13, simulated_13, correlated_13")
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(MS_13, exact_13, marker="o", label="independent, exact")
    ax.plot(MS_13, simulated_13, marker="x", ls="--", label="independent, simulated")
    ax.plot(MS_13, correlated_13, marker="s", label="c = 0.3, simulated")
    if filled(plateau_13):
        ax.axhline(plateau_13, color="grey", ls=":", label="your limit for c = 0.3")
    ax.axhline(Q_13, color="black", lw=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("number of voters M")
    ax.set_ylabel("probability that the majority is right")
    ax.legend()
    plt.show()

### Ex 14.14 — Voting et stacking avec les classifieurs du ch. 13 📦 ★★ ⏱️ 25 min
**Objectif :** comparer vote dur, vote souple et stacking, et voir ce que vaut une combinaison face à un seul bon modèle.  
**Prérequis :** Ex 14.11 · ch. 8 (validation croisée) · fiche §14.3 (voting et stacking) · **Fil rouge :** Penguins · **Parcours :** C

Pour que les membres fassent des erreurs **différentes**, chacun ne voit ici qu'**une** mesure : A un kNN à 15 voisins sur la longueur du bec, B `GaussianNB()` sur l'épaisseur du bec, C un arbre de profondeur 3 sur la nageoire (`MEMBERS_14`, une liste de paires `(nom, modèle)`, le format de scikit-learn). La cellule fournie affiche l'accuracy de chacun en validation croisée à 5 folds (`CV_14`, sur les 333 manchots et leurs 4 mesures `X_14` ; chaque membre choisit sa colonne).

Calcule, avec `cross_val_score(..., X_14, Y_14, cv=CV_14)` et la **moyenne** des 5 folds :
a) `hard_14` : `VotingClassifier(MEMBERS_14, voting="hard")` ;
b) `soft_14` : le même avec `voting="soft"` ;
c) `stack_14` : `StackingClassifier(MEMBERS_14, final_estimator=LogisticRegression(max_iter=1000))` (par défaut, son méta-modèle apprend sur les `predict_proba` des membres, obtenus par une validation croisée interne à 5 folds) ;
d) `single_14` : **un seul** modèle qui voit les 4 mesures, `make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))`.

Dans tes notes : classe les quatre et les trois membres. Pourquoi le vote souple bat-il le vote dur ici ? Que sait faire le méta-modèle que les deux votes ne savent pas ? Pourquoi un seul modèle sur les 4 mesures fait-il encore mieux, et quand un stacking vaut-il vraiment la peine ?

In [ ]:
X_14 = PENGUINS[MEASURES].to_numpy()
Y_14 = PENGUINS["species"].to_numpy()
CV_14 = StratifiedKFold(n_splits=5, shuffle=True, random_state=1414)


def on_column(column, model):
    """A pipeline that gives `model` only the column `column` of X."""
    return make_pipeline(ColumnTransformer([("keep", "passthrough", [column])]), model)


MEMBERS_14 = [("A", on_column(0, make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)))),   # bill length
              ("B", on_column(1, GaussianNB())),                                                           # bill depth
              ("C", on_column(2, DecisionTreeClassifier(max_depth=3, random_state=0)))]                    # flipper
for letter, member in MEMBERS_14:
    print(f"member {letter} alone: mean accuracy {cross_val_score(member, X_14, Y_14, cv=CV_14).mean():.3f}")

In [ ]:
hard_14 = ...     # a) mean accuracy (cross_val_score with CV_14) of VotingClassifier(MEMBERS_14, voting="hard")
soft_14 = ...     # b) the same with voting="soft"
stack_14 = ...    # c) the same for StackingClassifier(MEMBERS_14, final_estimator=LogisticRegression(max_iter=1000))
single_14 = ...   # d) the same for ONE model that sees the 4 measures:
                  #    make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

In [ ]:
wb.check("14.14a", hard_14)
wb.check("14.14b", soft_14)
wb.check("14.14c", stack_14)
wb.check("14.14d", single_14)

## Partie B · Bagging, forêts et ExtraTrees

Les forêts de scikit-learn sur California (14.15), le nombre d'arbres face à des colonnes de bruit (14.16), ton `BaggingClassifier` (14.17), ta forêt et tes ExtraTrees (14.18), puis l'importance des features (14.19). La cellule suivante charge California et la découpe comme le défi 14.28 : 16 512 districts d'entraînement, 4 128 de test (`random_state=42`).

In [ ]:
# California: the split of the challenge 14.28 (20 % of test districts, random_state=42), shared by parts B to D
CAL = wb.datasets.load_california()
CAL_FEATURES = [column for column in CAL.columns if column != "MedHouseVal"]
X_CAL, Y_CAL = CAL[CAL_FEATURES].to_numpy(), CAL["MedHouseVal"].to_numpy()
X_CAL_TRAIN, X_CAL_TEST, Y_CAL_TRAIN, Y_CAL_TEST = train_test_split(X_CAL, Y_CAL, test_size=0.2, random_state=42)
print(len(X_CAL_TRAIN), "training districts,", len(X_CAL_TEST), "test districts")

### Ex 14.15 — Forêts scikit-learn sur California : OOB et parallélisme 📦 ★★ ⏱️ 25 min
**Objectif :** utiliser une forêt de régression, son score out-of-bag et son parallélisme, et savoir ce qu'ils valent.  
**Prérequis :** Ex 14.11 · ch. 13 (arbres de régression) · ch. 9 (R²) · fiche §14.4 et §14.5 · **Fil rouge :** California · **Parcours :** R, C

`RandomForestRegressor` de scikit-learn sur 5 000 districts d'entraînement (`X_TRAIN_15`). En régression, `oob_score_` est un $R^2$ (ch. 9) calculé sur les exemples out-of-bag, et `score` rend aussi un $R^2$. Calcule :
a) `oob_15` : le `oob_score_` de `RandomForestRegressor(n_estimators=100, oob_score=True, random_state=0, n_jobs=-1)` ;
b) `test_15` : son $R^2$ sur les 4 128 districts de test ;
c) `oob_curve_15` : la liste des `oob_score_` de la même forêt pour chaque nombre d'arbres de `NS_15` (10 à 200) ;
d) `first_close_15` : le plus petit nombre d'arbres de `NS_15` dont le `oob_score_` est à moins de 0,01 du $R^2$ de test **de la même forêt**.

Avec 10 arbres, scikit-learn prévient que quelques districts n'ont aucun arbre qui ne les a pas vus : c'est normal (tu peux masquer l'avertissement avec `warnings.catch_warnings()`). La vérification contrôle a) à d), puis entraîne la forêt sur un cœur (`n_jobs=1`) et sur tous (`n_jobs=-1`), compare les durées et les prédictions.

Dans tes notes : pourquoi le score out-of-bag est-il pessimiste avec peu d'arbres ? Combien d'arbres prédisent un district out-of-bag, en moyenne, dans une forêt de 10 arbres ? Le gain de `n_jobs=-1` dépend de quoi ? Les prédictions sont-elles identiques au bit près, et pourquoi pourraient-elles ne pas l'être ?

In [ ]:
X_TRAIN_15, Y_TRAIN_15 = X_CAL_TRAIN[:5000], Y_CAL_TRAIN[:5000]   # 5,000 training districts
NS_15 = [10, 25, 50, 100, 200]

In [ ]:
forest_15 = ...       # RandomForestRegressor(n_estimators=100, oob_score=True, random_state=0, n_jobs=-1)
                      # fitted on X_TRAIN_15, Y_TRAIN_15
oob_15 = ...          # a) its out-of-bag R² (oob_score_)
test_15 = ...         # b) its R² on the test districts X_CAL_TEST, Y_CAL_TEST
oob_curve_15 = ...    # c) list: the oob_score_ of the same forest with n_estimators = n, for each n of NS_15
first_close_15 = ...  # d) the smallest n of NS_15 whose oob_score_ is within 0.01 of the test R² of the same forest

In [ ]:
wb.check("14.15a", oob_15)
wb.check("14.15b", test_15)
wb.check("14.15c", oob_curve_15)
wb.check("14.15d", first_close_15)
with wb.attempt("14.15"):
    if not filled(forest_15):
        raise NotImplementedError("forest_15")
    seconds_15, predictions_15 = {}, {}
    for n_jobs in [1, -1]:                                  # one core, then all the cores
        start_15 = time.perf_counter()
        model_15 = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=n_jobs).fit(X_TRAIN_15, Y_TRAIN_15)
        seconds_15[n_jobs] = time.perf_counter() - start_15
        predictions_15[n_jobs] = model_15.predict(X_CAL_TEST)
    print(f"{os.cpu_count()} core(s): fit in {seconds_15[1]:.2f} s with n_jobs=1, in {seconds_15[-1]:.2f} s with "
          f"n_jobs=-1 (speed-up × {seconds_15[1] / seconds_15[-1]:.1f})")
    print("same predictions to the last bit:", np.array_equal(predictions_15[1], predictions_15[-1]),
          "· largest difference:", float(np.max(np.abs(predictions_15[1] - predictions_15[-1]))))
    verdict("14.15", np.allclose(predictions_15[1], predictions_15[-1], rtol=0, atol=1e-9),
            "avec ou sans parallélisme, la même forêt (même random_state) donne les mêmes prédictions, à 1e-9 près.",
            "les prédictions changent avec n_jobs : est-ce bien la même forêt, avec le même random_state ?")

### Ex 14.16 — Combien d'arbres ? Bagging, forêt et ExtraTrees face au bruit 🔬 ★★ ⏱️ 30 min
**Objectif :** mesurer ce que le nombre d'arbres apporte à trois ensembles, et ce que des features de bruit leur coûtent.  
**Prérequis :** Ex 14.15 · ch. 7 (features de bruit) · fiche §14.4 à §14.6 · **Fil rouge :** synthétique · **Parcours :** C

Deux demi-lunes bruitées (500 points d'entraînement, 3 000 de test), puis les mêmes avec **30 colonnes de bruit** (`moons_16(n_noise)`). Trois ensembles de scikit-learn (`ENSEMBLES_16`, chacun appelé avec son nombre d'arbres) : un bagging d'arbres sans limite, une forêt aléatoire (`max_features="sqrt"`) et des ExtraTrees, tous avec `random_state=0`. Écris `curves_16(n_noise)`, qui rend, pour chaque ensemble, la liste de ses accuracies de test pour chaque nombre d'arbres de `NS_16` (1 à 400), puis :
a) `noisy_16` : `curves_16(30)` (la vérification lit la courbe d'ExtraTrees) ; et `clean_16` : `curves_16(0)` ;
b) `worst_16` : l'ensemble qui a la plus petite accuracy avec 400 arbres et 30 colonnes de bruit : `"A"` bagging, `"B"` forêt, `"C"` ExtraTrees ;
c) `gain_extra_16` : pour ExtraTrees avec 30 colonnes de bruit, l'accuracy à 400 arbres moins celle à 25 arbres ;
d) `drop_16` : `True` si l'une des six courbes perd plus de 0,005 entre 100 et 400 arbres, `False` sinon.

Ta cellule entraîne 54 ensembles (compte une quinzaine de secondes) ; la vérification trace les deux familles de courbes.

Dans tes notes : avec 30 colonnes de bruit, combien de features utiles un nœud de la forêt voit-il, en moyenne, parmi les 5 qu'il tire ? Pourquoi les seuils tirés au hasard d'ExtraTrees souffrent-ils encore plus du bruit ? Pourquoi 2 arbres font-ils souvent moins bien qu'un seul ? Combien d'arbres mettrais-tu ?

In [ ]:
NS_16 = [1, 2, 5, 10, 25, 50, 100, 200, 400]


def moons_16(n_noise):
    """Two noisy moons plus n_noise columns of standard normal noise: 500 training points, 3,000 test points."""
    X, y = wb.synth.make_moons(n=3500, noise=0.35, seed=1416)
    X = np.hstack([X, np.random.default_rng(1416).normal(size=(len(X), n_noise))])
    return X[:500], y[:500], X[500:], y[500:]


ENSEMBLES_16 = {"bagging": lambda n: BaggingClassifier(DecisionTreeClassifier(), n_estimators=n, random_state=0),
                "forest": lambda n: RandomForestClassifier(n_estimators=n, random_state=0),
                "extra": lambda n: ExtraTreesClassifier(n_estimators=n, random_state=0)}

In [ ]:
def curves_16(n_noise):
    """Return a dict {"bagging": [...], "forest": [...], "extra": [...]}: for each ensemble of ENSEMBLES_16 (called with
    the number of trees), the list of its test accuracies on moons_16(n_noise), one per number of trees of NS_16."""
    raise NotImplementedError("curves_16")


clean_16 = ...        # curves_16(0)
noisy_16 = ...        # a) curves_16(30)
worst_16 = ...        # b) the ensemble with the lowest test accuracy with 400 trees and 30 noise columns:
                      #    "A" bagging, "B" forest, "C" ExtraTrees
gain_extra_16 = ...   # c) ExtraTrees with 30 noise columns: accuracy with 400 trees minus accuracy with 25 trees
drop_16 = ...         # d) True if one of the six curves (clean_16 and noisy_16) loses more than 0.005 between 100 and
                      #    400 trees, False otherwise

In [ ]:
wb.check("14.16a", noisy_16.get("extra") if isinstance(noisy_16, dict) else noisy_16)
wb.check("14.16b", worst_16)
wb.check("14.16c", gain_extra_16)
wb.check("14.16d", drop_16)
with wb.attempt("14.16"):
    if not filled(clean_16, noisy_16):
        raise NotImplementedError("clean_16, noisy_16")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, (family, title, curves) in zip(axes, [("clean_16", "2 features (the moons)", clean_16),
                                                  ("noisy_16", "2 features + 30 noise columns", noisy_16)]):
        if not isinstance(curves, dict):
            print(f"❌ Ex 14.16 : {family} doit être un dictionnaire {{nom de l'ensemble: liste d'accuracies}}, pas un {type(curves).__name__}.")
            continue
        for name, curve in curves.items():
            if np.ndim(curve) != 1 or len(curve) != len(NS_16):
                print(f"❌ Ex 14.16 : {family}[{name!r}] doit être une liste de {len(NS_16)} accuracies, une par nombre "
                      f"d'arbres de NS_16 (forme reçue : {np.shape(curve)}).")
                continue
            ax.plot(NS_16, curve, marker="o", label=name)
        ax.set_xscale("log")
        ax.set_xlabel("number of trees")
        ax.set_title(title)
    axes[0].set_ylabel("test accuracy")
    axes[0].legend()
    plt.show()

### Ex 14.17 — Coder BaggingClassifier avec score out-of-bag 🔨 ★★★ ⏱️ 50 min
**Objectif :** écrire un bagging générique : des clones entraînés sur leurs bootstraps, un vote souple et un score out-of-bag.  
**Prérequis :** Ex 14.12 · Ex 13.25 · ch. 8 (`clone`) · fiche §14.4 (pseudo-code) · **Fil rouge :** Penguins · **mylearn :** `ensemble.py` · **Parcours :** C

> **Mode d'emploi mylearn** (comme aux chapitres précédents) : ouvre `mon_travail/mylearn/ensemble.py` (créé par `python tools/start_chapter.py 14`), lis la docstring de chaque fonction et de chaque méthode, remplace les `raise NotImplementedError(...)` par ton code et **enregistre**. NumPy seulement, plus tes modules des chapitres précédents : `clone` (ch. 8, `model_selection.py`) et tes arbres (ch. 13, `tree.py`) ; scikit-learn est l'**oracle** des tests. La cellule de vérification recharge ta librairie, compare quelques résultats à scikit-learn, puis lance les tests de l'exercice ; `python -m pytest tests/test_ch14_ensemble.py -q` les lance tous dans un terminal (compte quelques minutes si ton arbre est lent).

Écris la classe `BaggingClassifier`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `__init__` range ses six arguments tels quels ; `fit(X, y)` vérifie, apprend `classes_` (les labels distincts triés), `estimators_`, `estimators_samples_` (et `oob_score_` si `oob_score=True`), puis renvoie `self` ; un second `fit` du même objet redonne le même ensemble ;
- `fit` crée **un** générateur à partir de `random_state` (`np.random.default_rng(random_state)`), d'où vient tout le hasard de l'ensemble ; chaque membre a son propre échantillon d'indices, de la taille que fixe `max_samples` (un bootstrap, avec `bootstrap_indices`, si `bootstrap=True` ; sinon des indices **distincts**), est un `clone(gabarit)` neuf (le gabarit est `estimator`, ou `DecisionTreeClassifier()` de **ton** `tree.py` si `estimator` vaut `None` ; il n'est jamais entraîné lui-même), reçoit, s'il a un attribut `random_state`, un **entier** tiré de ce générateur, et apprend sur `X[indices]`, `y[indices]` ;
- `max_samples` : un entier de 1 à $n$, ou un réel $f$ dans $]0, 1]$ (`max(1, int(f * n))` tirages) ; `ValueError` sinon, ainsi que pour `n_estimators < 1`, `oob_score=True` avec `bootstrap=False`, un `X` qui n'est pas 2-D ou contient `NaN`, ou `len(X) != len(y)` ;
- `predict_proba(X)` : la moyenne des `predict_proba` des membres, **alignés sur `classes_`** (un membre qui n'a vu que certaines classes donne une probabilité nulle aux autres) ; `predict(X)` : la classe de plus grande probabilité moyenne ; `score(X, y)` : l'accuracy, un `float` Python ; `RuntimeError` si `predict_proba`, `predict` ou `score` est appelée avant `fit` ;
- `oob_score_` : chaque exemple d'entraînement est prédit par la moyenne des probabilités des **seuls** membres qui ne l'ont pas vu ; les exemples vus par tous sont laissés de côté ; `ValueError` si aucun exemple n'est out-of-bag.

La vérification essaie l'exemple de la docstring, compare ton bagging (25 arbres, 5 graines) à celui de scikit-learn sur deux mesures des manchots, contrôle les échantillons, les clones et la reproductibilité, puis lance les tests.

Dans tes notes : pourquoi un clone neuf par membre, plutôt que le gabarit réentraîné 25 fois ? Pourquoi le vote souple doit-il aligner les colonnes sur `classes_` ? Que coûte le score out-of-bag ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")   # reload your saved file
with checked("14.17"):
    X_doc_17 = np.array([[0.0], [1.0], [2.0], [3.0], [10.0], [11.0], [12.0], [13.0]])
    y_doc_17 = np.array([0, 0, 0, 0, 1, 1, 1, 1])
    doc_17 = fitted(mylearn.ensemble.BaggingClassifier(n_estimators=5, random_state=0), X_doc_17, y_doc_17)
    predicted_doc_17 = doc_17.predict(np.array([[1.5], [11.5]]))
    if returned("14.17", "predict", predicted_doc_17):
        print("docstring example:", len(doc_17.estimators_), "members, predict", np.asarray(predicted_doc_17).tolist(),
              "(expected 5 members and [0, 1])")
with checked("14.17"):
    X_17, y_17 = P_TRAIN[["bill_depth_mm", "flipper_length_mm"]].to_numpy(), P_TRAIN["species"].to_numpy()
    X_test_17, y_test_17 = P_TEST[["bill_depth_mm", "flipper_length_mm"]].to_numpy(), P_TEST["species"].to_numpy()
    mine_17, oracle_17 = [], []
    start_17 = time.perf_counter()
    for seed in range(5):
        model = fitted(mylearn.ensemble.BaggingClassifier(n_estimators=25, oob_score=True, random_state=seed), X_17, y_17)
        mine_17.append((model.score(X_test_17, y_test_17), model.oob_score_))
    seconds_17 = time.perf_counter() - start_17
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")              # scikit-learn warns when a penguin is in every bootstrap
        for seed in range(5):
            model = BaggingClassifier(DecisionTreeClassifier(), n_estimators=25, oob_score=True, random_state=seed).fit(X_17, y_17)
            oracle_17.append((model.score(X_test_17, y_test_17), model.oob_score_))
    (acc_17, oob_17), (sk_acc_17, sk_oob_17) = np.mean(mine_17, axis=0), np.mean(oracle_17, axis=0)
    print(f"5 baggings of 25 trees (2 measures of the penguins), mean over random_state 0 to 4:\n"
          f"  yours:        test accuracy {acc_17:.3f}, oob_score_ {oob_17:.3f} ({seconds_17:.1f} s for the 5)\n"
          f"  scikit-learn: test accuracy {sk_acc_17:.3f}, oob_score_ {sk_oob_17:.3f}")
    verdict("14.17", abs(acc_17 - sk_acc_17) < 0.03 and abs(oob_17 - sk_oob_17) < 0.05,
            "ton bagging est à moins de 3 points de scikit-learn en test, et ton score out-of-bag à moins de 5 points.",
            "ton bagging s'écarte de scikit-learn de plus de 3 points en test, ou ton score out-of-bag de plus de 5 : "
            "les tests ci-dessous disent où.")
with checked("14.17"):
    template_17 = mylearn.tree.DecisionTreeClassifier(max_depth=3)
    bag_17 = fitted(mylearn.ensemble.BaggingClassifier(template_17, n_estimators=10, random_state=1), X_17, y_17)
    again_17 = fitted(mylearn.ensemble.BaggingClassifier(template_17, n_estimators=10, random_state=1), X_17, y_17)
    samples_17 = [np.asarray(rows) for rows in bag_17.estimators_samples_]
    checks_17 = {
        "10 membres, chacun avec ses 233 indices tirés avec remise": len(bag_17.estimators_) == 10
            and all(rows.shape == (233,) and len(np.unique(rows)) < 233 for rows in samples_17),
        "des échantillons différents d'un membre à l'autre": len({tuple(rows) for rows in samples_17}) == 10,
        "le gabarit n'est jamais entraîné lui-même (des clones)": getattr(template_17, "root_", None) is None
            and all(member is not template_17 for member in bag_17.estimators_),
        "les mêmes probabilités pour le même random_state": np.allclose(bag_17.predict_proba(X_test_17),
                                                                       again_17.predict_proba(X_test_17)),
    }
    for what, ok in checks_17.items():
        verdict("14.17", ok, what + ".", "attendu : " + what + " ; les tests ci-dessous disent où.")
if written(lambda: mylearn.ensemble.BaggingClassifier(n_estimators=1).fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("ensemble", keyword="test_bagging_classifier_")
else:
    print("⏳ Ex 14.17 : les tests attendent le fit de ton BaggingClassifier.")

### Ex 14.18 — Coder RandomForest et ExtraTrees 🔨 ★★★ ⏱️ 45 min
**Objectif :** construire une forêt aléatoire et des ExtraTrees sur ton bagging et tes arbres, avec leurs importances.  
**Prérequis :** Ex 14.17 · Ex 13.25 (`max_features`, `splitter`) · fiche §14.5 et §14.6 · **Fil rouge :** synthétique · **mylearn :** `ensemble.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/ensemble.py`, mêmes règles qu'en 14.17. Enregistre, puis relance la cellule de vérification.

Écris les classes `RandomForestClassifier` et `ExtraTreesClassifier`. **Contrat** (les docstrings, plus ces précisions que vérifient les tests) :
- chaque arbre est un `DecisionTreeClassifier` de **ton** `tree.py`, avec les `max_depth`, `min_samples_split`, `min_samples_leaf` et `max_features` de l'ensemble, plus `splitter="random"` pour ExtraTrees ; il apprend sur un bootstrap de $n$ tirages (`bootstrap=True`) ou sur tout l'entraînement (`bootstrap=False`), et reçoit un `random_state` entier, différent d'un arbre à l'autre, tiré d'un générateur créé dans `fit` à partir du `random_state` de l'ensemble (comme les membres de ton bagging) ; `__init__` range ses arguments tels quels, et `fit` n'ajoute que des attributs appris (un nom qui finit par `_`) ou privés (un nom qui commence par `_`) ;
- après `fit` : `estimators_`, `classes_`, `feature_importances_` (et `oob_score_` pour la forêt si `oob_score=True`) ; `predict_proba`, `predict` et `score` comme le bagging (vote souple, `RuntimeError` avant `fit`) ; `ValueError` pour un hyperparamètre invalide, `NaN`, `len(X) != len(y)`, ou `oob_score=True` avec `bootstrap=False` ;
- `feature_importances_` : la moyenne des `feature_importances_` des arbres **qui coupent au moins une fois**, renormalisée pour sommer à 1 (des zéros si aucun arbre ne coupe), comme scikit-learn : un arbre réduit à une feuille ne donne que des zéros, qui ne doivent pas tirer la moyenne vers le bas ; c'est ce que demande l'exemple de la docstring (`array([1.])`), que la seule moyenne ne donne pas.

La vérification essaie les exemples des docstrings, compare tes deux ensembles (20 arbres, 3 graines) à ceux de scikit-learn sur des demi-lunes avec 3 colonnes de bruit (accuracy et importances), vérifie les hyperparamètres des arbres, puis lance les tests (les tests statistiques entraînent beaucoup de forêts : compte une à deux minutes si ton arbre est lent).

Dans tes notes : combien de lignes ta forêt a-t-elle demandé, une fois le bagging écrit ? Lequel de tes deux ensembles s'entraîne le plus vite, et pourquoi ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")   # reload your saved file
with checked("14.18"):
    X_doc_18 = np.array([[0.0], [1.0], [2.0], [3.0], [10.0], [11.0], [12.0], [13.0]])
    y_doc_18 = np.array([0, 0, 0, 0, 1, 1, 1, 1])
    forest_doc_18 = fitted(mylearn.ensemble.RandomForestClassifier(n_estimators=5, random_state=0), X_doc_18, y_doc_18)
    extra_doc_18 = fitted(mylearn.ensemble.ExtraTreesClassifier(n_estimators=5, random_state=0), X_doc_18, y_doc_18)
    print("docstring examples: forest", np.asarray(forest_doc_18.predict(np.array([[1.5], [11.5]]))).tolist(),
          "importances", np.round(np.asarray(forest_doc_18.feature_importances_, dtype=float), 4).tolist(), "· extra",
          np.asarray(extra_doc_18.predict(np.array([[1.5], [11.5]]))).tolist(), "(expected [0, 1], [1.0] and [0, 1])")
with checked("14.18"):
    X_18, y_18 = wb.synth.make_moons(n=1300, noise=0.3, seed=1418)
    X_18 = np.hstack([X_18, np.random.default_rng(1418).normal(size=(1300, 3))])   # 2 useful features, 3 of noise
    X_train_18, y_train_18, X_test_18, y_test_18 = X_18[:300], y_18[:300], X_18[300:], y_18[300:]
    pairs_18 = [("RandomForestClassifier", mylearn.ensemble.RandomForestClassifier, RandomForestClassifier),
                ("ExtraTreesClassifier", mylearn.ensemble.ExtraTreesClassifier, ExtraTreesClassifier)]
    for name, mine_class, oracle_class in pairs_18:
        start_18 = time.perf_counter()
        mine_18 = [fitted(mine_class(n_estimators=20, random_state=seed), X_train_18, y_train_18) for seed in range(3)]
        seconds_18 = time.perf_counter() - start_18
        oracles_18 = [oracle_class(n_estimators=20, random_state=seed).fit(X_train_18, y_train_18) for seed in range(3)]
        acc_18 = np.mean([model.score(X_test_18, y_test_18) for model in mine_18])
        sk_acc_18 = np.mean([model.score(X_test_18, y_test_18) for model in oracles_18])
        importances_18 = np.mean([np.asarray(model.feature_importances_, dtype=float) for model in mine_18], axis=0)
        print(f"{name}, 20 trees, mean over random_state 0 to 2: yours {acc_18:.3f} ({seconds_18:.1f} s), "
              f"scikit-learn {sk_acc_18:.3f}; your importances {np.round(importances_18, 3).tolist()}")
        verdict("14.18", abs(acc_18 - sk_acc_18) < 0.03,
                f"{name} : à moins de 3 points de scikit-learn.",
                f"{name} : plus de 3 points d'écart avec scikit-learn ; les tests ci-dessous disent où.")
        verdict("14.18", importances_18.shape == (5,) and set(np.argsort(importances_18)[-2:]) == {0, 1}
                and abs(importances_18.sum() - 1) < 1e-6,
                f"{name} : les deux features utiles (0 et 1) ont les plus grandes importances, qui somment à 1.",
                f"{name} : attendu des importances de forme (5,), qui somment à 1, les plus grandes pour les features 0 "
                "et 1 (les trois autres sont du bruit).")
    trees_18 = {"forest": fitted(mylearn.ensemble.RandomForestClassifier(n_estimators=3, max_depth=4, random_state=0),
                                 X_train_18, y_train_18).estimators_[0],
                "extra": fitted(mylearn.ensemble.ExtraTreesClassifier(n_estimators=3, max_depth=4, random_state=0),
                                X_train_18, y_train_18).estimators_[0]}
    verdict("14.18", getattr(trees_18["forest"], "max_features", None) == "sqrt"
            and getattr(trees_18["forest"], "max_depth", None) == 4 and getattr(trees_18["extra"], "splitter", None) == "random",
            "les arbres reçoivent les hyperparamètres de l'ensemble (max_features=\"sqrt\", max_depth, et "
            "splitter=\"random\" pour ExtraTrees).",
            "chaque arbre doit recevoir max_depth, min_samples_split, min_samples_leaf et max_features de l'ensemble, "
            "et splitter=\"random\" pour ExtraTrees.")
if written(lambda: mylearn.ensemble.RandomForestClassifier(n_estimators=1).fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("ensemble", keyword="test_random_forest_classifier_ or test_extra_trees_classifier_")
else:
    print("⏳ Ex 14.18 : les tests attendent le fit de ta RandomForestClassifier.")

### Ex 14.19 — Importance des features : impureté contre permutation 📦 ★★★ ⏱️ 35 min
**Objectif :** comparer l'importance MDI d'une forêt à l'importance par permutation, et voir ce qui trompe la première.  
**Prérequis :** Ex 14.15 · ch. 12 (sélection de features) · fiche §14.5 (importance des features) · **Fil rouge :** California · **Parcours :** C

California, avec deux colonnes **sans aucun lien** avec la cible : `row_id`, un numéro différent pour chaque district, et `coin`, 0 ou 1 au hasard. Une forêt de 100 arbres apprend sur 5 000 districts (`X_TRAIN_19`, un DataFrame) ; 2 000 autres districts d'entraînement servent de validation (`X_VAL_19`). Calcule `forest_19`, `mdi_19` (son `feature_importances_`) et `perm_19` (l'importance par permutation sur la validation, avec `permutation_importance(..., n_repeats=5, random_state=0, n_jobs=-1)`, attribut `importances_mean`), deux `pandas.Series` indexées par `FEATURES_19` et triées par ordre décroissant, puis :
a) `rank_id_19` : le rang de `row_id` dans `mdi_19` (1 pour la plus importante) ;
b) `ratio_19` : l'importance MDI de `row_id` divisée par celle de `coin` ;
c) `second_mdi_19` : le nom de la deuxième feature de `mdi_19` ;
d) `second_perm_19` : le nom de la deuxième feature de `perm_19` ;
e) `id_train_19` : l'importance par permutation de `row_id` mesurée sur les districts d'**entraînement** (le même appel sur `X_TRAIN_19`, `Y_TRAIN_19`) ;
f) `joint_19` : la baisse moyenne du $R^2$ de validation quand on permute **ensemble** `Latitude` et `Longitude` : pour `r` de 0 à 4, une permutation `np.random.default_rng(r).permutation(len(X_VAL_19))`, appliquée aux deux colonnes d'une copie de `X_VAL_19` ; la baisse est le $R^2$ de validation (`forest_19.score`) moins celui de la copie.

La vérification contrôle a) à f), puis trace les deux importances.

Dans tes notes : pourquoi `row_id` pèse-t-il tant plus que `coin` dans l'importance MDI ? Que dit e) sur ce que la forêt a appris de `row_id` ? Compare f) à la somme des importances par permutation de `Latitude` et de `Longitude` : pourquoi n'est-ce pas la même chose ?

In [ ]:
rng_19 = np.random.default_rng(1419)
DATA_19 = pd.concat([pd.DataFrame({"row_id": rng_19.permutation(len(CAL)).astype(float),   # a distinct number per
                                   "coin": rng_19.integers(0, 2, len(CAL)).astype(float)}),  # district; 0 or 1 at random:
                     CAL[CAL_FEATURES]], axis=1)                                             # both unrelated to y
FEATURES_19 = list(DATA_19.columns)
rows_train_19, _ = train_test_split(np.arange(len(DATA_19)), test_size=0.2, random_state=42)   # the split of part B
X_TRAIN_19, Y_TRAIN_19 = DATA_19.iloc[rows_train_19[:5000]], Y_CAL[rows_train_19[:5000]]
X_VAL_19, Y_VAL_19 = DATA_19.iloc[rows_train_19[5000:7000]], Y_CAL[rows_train_19[5000:7000]]
print(FEATURES_19)

In [ ]:
forest_19 = ...      # RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1) fitted on X_TRAIN_19, Y_TRAIN_19
mdi_19 = ...         # pandas Series: forest_19.feature_importances_, indexed by FEATURES_19, sorted in decreasing order
perm_19 = ...        # pandas Series: the mean permutation importances on X_VAL_19, Y_VAL_19
                     #   (permutation_importance(forest_19, X_VAL_19, Y_VAL_19, n_repeats=5, random_state=0, n_jobs=-1)),
                     #   indexed by FEATURES_19, sorted in decreasing order
rank_id_19 = ...     # a) the rank of "row_id" in mdi_19 (1 = the most important)
ratio_19 = ...       # b) the MDI importance of "row_id" divided by that of "coin"
second_mdi_19 = ...  # c) the name of the second most important feature in mdi_19
second_perm_19 = ... # d) the name of the second most important feature in perm_19
id_train_19 = ...    # e) the mean permutation importance of "row_id" on the TRAINING districts (same call on X_TRAIN_19)
joint_19 = ...       # f) the mean drop of R² on X_VAL_19 when Latitude and Longitude are permuted TOGETHER (see the statement)

In [ ]:
wb.check("14.19a", rank_id_19)
wb.check("14.19b", ratio_19)
wb.check("14.19c", second_mdi_19)
wb.check("14.19d", second_perm_19)
wb.check("14.19e", id_train_19)
wb.check("14.19f", joint_19)
with wb.attempt("14.19"):
    if not filled(mdi_19, perm_19):
        raise NotImplementedError("mdi_19, perm_19")
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    pd.Series(mdi_19).sort_values().plot.barh(ax=axes[0], title="MDI (training)")
    pd.Series(perm_19).sort_values().plot.barh(ax=axes[1], title="permutation (validation, drop of R²)")
    plt.tight_layout()
    plt.show()

## Partie C · Le boosting

Lire les courbes d'entraînement et de validation d'un gradient boosting (14.20), prévoir l'effet du learning rate (14.21), voir AdaBoost découper le plan souche après souche (14.22), réparer un AdaBoost qui n'apprend rien (14.23), puis tes `AdaBoostClassifier` (14.24) et `GradientBoostingRegressor` (14.25). La cellule suivante prend 3 000 districts d'entraînement pour les boostings et 2 000 autres pour les valider, et définit `boosting_curves`, qui rend les RMSE d'entraînement et de validation après chaque arbre.

In [ ]:
# 3,000 training districts to fit the boostings, 2,000 others to validate them (taken from the training split)
X_FIT_C, Y_FIT_C = X_CAL_TRAIN[:3000], Y_CAL_TRAIN[:3000]
X_VAL_C, Y_VAL_C = X_CAL_TRAIN[3000:5000], Y_CAL_TRAIN[3000:5000]


def boosting_curves(**params):
    """Fit GradientBoostingRegressor(random_state=0, **params) on X_FIT_C; return the training RMSE and the validation
    RMSE after each tree (two arrays)."""
    model = GradientBoostingRegressor(random_state=0, **params).fit(X_FIT_C, Y_FIT_C)
    validation = np.array([rmse(Y_VAL_C, predicted) for predicted in model.staged_predict(X_VAL_C)])
    return np.sqrt(model.train_score_), validation

### Ex 14.20 — Lire les courbes train/validation d'un boosting 📈 ★ ⏱️ 15 min
**Objectif :** diagnostiquer un gradient boosting à ses courbes d'entraînement et de validation, et choisir où s'arrêter.  
**Prérequis :** ✏️ 14.7 · ch. 9 (overfitting, underfitting) · fiche, section sur le gradient boosting · **Fil rouge :** California · **Parcours :** R, C

La cellule fournie entraîne trois `GradientBoostingRegressor` sur 3 000 districts, avec 400 arbres et un learning rate de 0,3. Ils ne diffèrent que par la **profondeur** de leurs arbres : 1, 3 et 8, dans un ordre caché (panneaux A, B et C). Elle trace, pour chacun, la RMSE d'entraînement et celle de validation après chaque arbre. Réponds en lisant les courbes, puis vérifie d) et e) par le calcul :
a) `overfit_20` : le panneau dont le modèle apprend son entraînement **par cœur** ;
b) `underfit_20` : le panneau dont le modèle est encore en **underfitting** après 400 arbres ;
c) `stop_20` : pour le panneau restant (ni celui de a), ni celui de b)), où est le minimum de la courbe de validation : `"A"` avant 50 arbres ; `"B"` de 50 à 300 arbres ; `"C"` de 301 à 400 arbres ; `"D"` la courbe de validation descend encore à 400 arbres ;
d) `best_trees_20` : pour les panneaux A, B et C, le nombre d'arbres (de 1 à 400) au minimum de la RMSE de validation, calculé à partir de `CURVES_20` (un dictionnaire `panneau → (RMSE d'entraînement, RMSE de validation)`) ;
e) `gap_20` : pour le panneau de c), la RMSE de validation après 400 arbres moins son minimum.

Dans tes notes : attribue une profondeur à chaque panneau, et dis à quoi tu la reconnais. Combien d'arbres garderais-tu pour chacun ? Lequel des trois choisirais-tu, et que changerais-tu à son réglage ?

In [ ]:
ORDER_20 = np.random.default_rng(1421).permutation([1, 3, 8])   # the depths of panels A, B and C, in a hidden order
CURVES_20 = {panel: boosting_curves(max_depth=int(depth), learning_rate=0.3, n_estimators=400)
             for panel, depth in zip("ABC", ORDER_20)}                 # panel -> (training RMSE, validation RMSE)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, (panel, (train, validation)) in zip(axes, CURVES_20.items()):
    ax.plot(np.arange(1, 401), train, label="training")
    ax.plot(np.arange(1, 401), validation, label="validation")
    ax.set_xlabel("number of trees")
    ax.set_title(f"panel {panel}")
axes[0].set_ylabel("RMSE")
axes[0].legend()
plt.show()
del ORDER_20                         # the answer is not kept in the notebook

In [ ]:
overfit_20 = ...      # a) the panel ("A", "B" or "C") whose model learns its training set by heart
underfit_20 = ...     # b) the panel whose model still underfits after 400 trees
stop_20 = ...         # c) for the remaining panel (neither a) nor b)), where is the minimum of the validation curve?
                      #    "A" before 50 trees · "B" from 50 to 300 trees · "C" from 301 to 400 trees ·
                      #    "D" the validation curve still goes down at 400 trees
best_trees_20 = ...   # d) list: for panels A, B and C, the number of trees (1 to 400) at the minimum of the validation
                      #    RMSE, computed from CURVES_20
gap_20 = ...          # e) for the panel of c): the validation RMSE after 400 trees minus its minimum

In [ ]:
wb.check("14.20a", overfit_20)
wb.check("14.20b", underfit_20)
wb.check("14.20c", stop_20)
wb.check("14.20d", best_trees_20)
wb.check("14.20e", gap_20)

### Ex 14.21 — Learning rate et nombre d'arbres : prédire les courbes 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir, en nombres, ce que le learning rate change au nombre d'arbres utile et à l'erreur de validation.  
**Prérequis :** Ex 14.20 · fiche, section sur le gradient boosting (le learning rate et le nombre d'arbres) · **Fil rouge :** California · **Parcours :** C

Mêmes données que 14.20, des arbres de profondeur 3, jusqu'à 1 000 arbres, et cinq learning rates : 0,02 ; 0,05 ; 0,1 ; 0,5 et 1. La fiche te dit déjà le sens des effets (un petit learning rate demande plus d'arbres et généralise d'ordinaire mieux) ; prédis plutôt leur **taille**.

Prédis, **avant** d'exécuter quoi que ce soit :
a) `ratio_21` : en passant de 0,5 à 0,05 (un learning rate divisé par 10), le nombre d'arbres au minimum de la RMSE de validation est multiplié par environ : `"A"` 1 ; `"B"` 3 ; `"C"` 10 ; `"D"` 100 ;
b) `gain_21` : la meilleure RMSE de validation avec 0,05, comparée à celle avec 0,5 : `"A"` moins bonne ; `"B"` meilleure de moins de 0,005 ; `"C"` meilleure de 0,005 à 0,1 ; `"D"` meilleure de plus de 0,1 ;
c) `slow_beats_21` : avec 0,02, la meilleure RMSE de validation **en 1 000 arbres au plus** est-elle plus basse qu'avec 0,1 ? (`True` ou `False`)

Puis exécute l'expérience (une quarantaine de secondes) : elle affiche, pour chaque learning rate, le nombre d'arbres au minimum, la meilleure RMSE de validation, la RMSE après 1 000 arbres et la RMSE d'entraînement, et trace les courbes de validation.

In [ ]:
LRS_21 = [0.02, 0.05, 0.1, 0.5, 1.0]   # learning rates, all with trees of depth 3 and up to 1,000 trees

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
ratio_21 = ...        # a) "A", "B", "C" or "D"
gain_21 = ...         # b) "A", "B", "C" or "D"
slow_beats_21 = ...   # c) True or False

In [ ]:
if any(answer is ... or answer is None for answer in [ratio_21, gain_21, slow_beats_21]):
    print("⏳ Ex 14.21 : écris d'abord tes trois prédictions, puis relance cette cellule.")
else:
    curves_21 = {lr: boosting_curves(max_depth=3, learning_rate=lr, n_estimators=1000) for lr in LRS_21}   # ≈ 40 s
    print(" learning rate · trees at the minimum · best validation RMSE · after 1,000 trees · training RMSE after 1,000 trees")
    for lr, (train, validation) in curves_21.items():
        print(f"{lr:14.2f} · {int(np.argmin(validation)) + 1:19d} · {validation.min():24.4f} · {validation[-1]:17.4f} · "
              f"{train[-1]:.4f}")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for lr, (train, validation) in curves_21.items():
        ax.plot(np.arange(1, 1001), validation, label=f"learning rate {lr}")
    ax.set_xscale("log")
    ax.set_ylim(0.5, 0.9)
    ax.set_xlabel("number of trees")
    ax.set_ylabel("validation RMSE")
    ax.legend()
    plt.show()
    wb.check("14.21a", ratio_21)
    wb.check("14.21b", gain_21)
    wb.check("14.21c", slow_beats_21)

Dans tes notes : compare avec tes prédictions. Le produit « learning rate × nombre d'arbres au minimum » est-il à peu près constant ? Que se passe-t-il à droite du minimum pour les grands learning rates, et pour 0,02 ? Quel couple (learning rate, nombre d'arbres) choisirais-tu, avec quel budget de temps ?

### Ex 14.22 — Frontières d'AdaBoost après 1, 3, 10 et 50 souches 🎨 ★★ ⏱️ 25 min
**Objectif :** voir un ensemble de souches découper le plan, tour après tour.  
**Prérequis :** ✏️ 14.6 · ch. 13 (souches, frontières) · fiche §14.7 (AdaBoost) · **Fil rouge :** synthétique · **Parcours :** C

Deux demi-lunes bruitées (300 points, `X_22`) et l'`AdaBoostClassifier` de scikit-learn, dont les membres par défaut sont des souches. Écris :
- `model_22` : `AdaBoostClassifier(n_estimators=50, random_state=0)` entraîné sur `X_22`, `Y_22` ;
- `stage_predictions_22(model, X, rounds)` : un dictionnaire qui associe à chaque $m$ de `rounds` les prédictions du modèle sur `X` quand il ne garde que ses $m$ premiers membres (`model.staged_predict(X)` donne celles après 1, 2, 3… membres) ;
- `draw_22(model, X, y, rounds)` : une figure de `len(rounds)` panneaux ; dans le panneau $m$, les régions prédites sur la grille `GRID_22` après $m$ membres (remises à la forme de `GRID1_22`, avec `ax.contourf`), les points d'entraînement colorés par classe, et un titre avec $m$ et l'accuracy d'entraînement après $m$ membres ; elle rend la figure ;
- `train_acc_22` : la liste des accuracies d'entraînement après 1, 3, 10 et 50 membres (`ROUNDS_22`).

La vérification contrôle `train_acc_22`, vérifie qu'après une souche les régions sont coupées par une seule droite parallèle à un axe, puis appelle `draw_22`.

Dans tes notes : décris la frontière après 3 souches, puis après 50. Pourquoi est-elle toujours faite de segments horizontaux et verticaux ? Où l'ensemble finit-il par dessiner de petits rectangles, et est-ce bon signe ?

In [ ]:
X_22, Y_22 = wb.synth.make_moons(n=300, noise=0.25, seed=1422)
ROUNDS_22 = [1, 3, 10, 50]
GRID1_22, GRID2_22 = np.meshgrid(np.linspace(-1.6, 2.6, 160), np.linspace(-1.2, 1.7, 110))
GRID_22 = np.column_stack([GRID1_22.ravel(), GRID2_22.ravel()])   # 17,600 points that cover the moons

In [ ]:
model_22 = ...   # AdaBoostClassifier(n_estimators=50, random_state=0) fitted on X_22, Y_22 (its members are stumps)


def stage_predictions_22(model, X, rounds):
    """Return a dict {m: the predictions of the fitted AdaBoost `model` on X when it keeps only its first m members},
    for each m of `rounds` (use model.staged_predict, which yields the predictions after 1, 2, 3... members)."""
    raise NotImplementedError("stage_predictions_22")


def draw_22(model, X, y, rounds):
    """Draw 1 x len(rounds) panels: in panel m, the regions predicted after m members on GRID_22 (reshaped like
    GRID1_22, e.g. with ax.contourf), the training points X coloured by y, and a title with m and the training
    accuracy after m members. Return the figure."""
    raise NotImplementedError("draw_22")


train_acc_22 = ...   # list: the training accuracy of model_22 after 1, 3, 10 and 50 members

In [ ]:
wb.check("14.22a", train_acc_22)
with wb.attempt("14.22"):
    if not filled(model_22):
        raise NotImplementedError("model_22")
    stages_22 = stage_predictions_22(model_22, GRID_22, ROUNDS_22)
    if returned("14.22", "stage_predictions_22", stages_22) and not (isinstance(stages_22, dict) and set(ROUNDS_22) <= set(stages_22)):
        print("❌ Ex 14.22 : stage_predictions_22 doit rendre un dictionnaire {m: prédictions} avec une clé pour chaque m "
              f"de ROUNDS_22 ({ROUNDS_22}).")
    elif stages_22 is not None:
        first_22 = np.asarray(stages_22[1]).reshape(GRID1_22.shape)
        one_cut_22 = (first_22 == first_22[:1, :]).all() or (first_22 == first_22[:, :1]).all()
        verdict("14.22", one_cut_22 and np.array_equal(np.asarray(stages_22[50]), model_22.predict(GRID_22)),
                "après une souche, une seule coupure parallèle à un axe ; après 50, les prédictions de model_22.",
                "attendu : après 1 membre, des régions coupées par une seule droite verticale ou horizontale (une "
                "souche), et après 50 membres les prédictions de model_22.predict ; relis staged_predict.")
        figure_22 = draw_22(model_22, X_22, Y_22, ROUNDS_22)
        if returned("14.22", "draw_22", figure_22):
            plt.show()

### Ex 14.23 — L'AdaBoost qui n'apprend rien 🐛 ★★ ⏱️ 25 min
**Objectif :** trouver et corriger les erreurs d'un AdaBoost écrit à la main, en s'aidant de ses propriétés.  
**Prérequis :** ✏️ 14.6 · fiche §14.7 (pseudo-code d'AdaBoost) · **Fil rouge :** synthétique · **Parcours :** C

Un collègue a écrit `adaboost_23`, un AdaBoost (SAMME, deux classes 0 et 1) avec des souches de scikit-learn, et `predict_23`, son vote pondéré (correct). Son accuracy d'entraînement ne bouge pas, quel que soit le nombre de tours. Il y a **trois** erreurs dans `adaboost_23` : trouve-les et corrige-les (sans changer sa signature ni ce qu'elle rend), en t'aidant du pseudo-code de la fiche (§14.7).

La vérification lance ta version sur 400 points de deux demi-lunes, puis contrôle trois propriétés, dans l'ordre où les erreurs se masquent (un point ne se teste que si les précédents passent) : 1) après le premier tour, la mise à jour alourdit les exemples que la souche a mal classés, par rapport aux autres ; 2) les poids avec lesquels chaque souche est entraînée somment à 1 ; 3) les souches changent d'un tour à l'autre. Quand les trois points passent, elle vérifie a) l'accuracy d'entraînement des 40 tours et b) le poids $\alpha$ de la **deuxième** souche, puis compare ta version à l'`AdaBoostClassifier` de scikit-learn.

Dans tes notes : pour chaque erreur, le symptôme qu'elle provoque seule. Lesquelles des trois erreurs, seules, figent l'accuracy ? Une fois tout corrigé, pourquoi les exemples que la souche précédente classait mal pèsent-ils exactement la moitié du total après chaque mise à jour (à deux classes, avec $\alpha = \ln\frac{1 - \varepsilon}{\varepsilon}$) ?

In [ ]:
X_23, Y_23 = wb.synth.make_moons(n=400, noise=0.3, seed=1423)   # two classes, 0 and 1


def predict_23(stumps, alphas, X):
    """Weighted vote of the stumps (correct, nothing to fix): the class whose stumps have the largest sum of alphas."""
    votes = np.zeros((len(X), 2))
    for stump, alpha in zip(stumps, alphas):
        votes[np.arange(len(X)), stump.predict(X)] += alpha
    return np.argmax(votes, axis=1)

In [ ]:
# A colleague's AdaBoost (SAMME, two classes 0 and 1): its accuracy does not improve, whatever the number of rounds.
# Find and fix its bugs (there are three), then rerun the check cell.
def adaboost_23(X, y, n_rounds=40):
    """AdaBoost with scikit-learn stumps. Return (stumps, alphas, weights), where weights[m] is the vector of sample
    weights (summing to 1) with which stump m was fitted."""
    n = len(X)
    w = np.full(n, 1 / n)
    stumps, alphas, weights = [], [], []
    for m in range(n_rounds):
        weights.append(w.copy())
        stump = DecisionTreeClassifier(max_depth=1, random_state=m)
        stump.fit(X, y)
        wrong = stump.predict(X) != y
        err = np.sum(w[wrong])
        alpha = np.log((1 - err) / err)
        w = w * np.exp(alpha * (stump.predict(X) == y))
        stumps.append(stump)
        alphas.append(alpha)
    return stumps, alphas, weights

In [ ]:
with checked("14.23"):
    try:
        with np.errstate(all="ignore"):
            result_23 = adaboost_23(X_23, Y_23)
        crash_23 = None
    except ValueError as exc:                            # NaN weights refused by scikit-learn
        if "nan" not in str(exc).lower() and "infinity" not in str(exc).lower():
            raise
        result_23, crash_23 = None, exc
    if crash_23 is not None:
        print("❌ Ex 14.23 : point 1 : les poids sont devenus NaN ou infinis (scikit-learn refuse alors d'entraîner la "
              "souche) : l'erreur pondérée est tombée à 0 et α est devenu infini. Relis quels exemples la mise à jour "
              "doit alourdir.")
        print("⏳ Ex 14.23 : points 2 et 3, puis a) et b) : corrige d'abord le point 1.")
    elif returned("14.23", "adaboost_23", result_23):
        stumps_23, alphas_23, weights_23 = result_23
        weights_23 = [np.asarray(w, dtype=float) for w in weights_23]
        wrong0_23 = stumps_23[0].predict(X_23) != Y_23
        with np.errstate(all="ignore"):
            growth_23 = weights_23[1] / weights_23[0]        # how the update changed each weight after round 1
        cuts_23 = {(int(s.tree_.feature[0]), round(float(s.tree_.threshold[0]), 6)) for s in stumps_23}
        diagnostics_23 = [
            ("après le premier tour, la mise à jour alourdit les exemples que la souche a mal classés, par rapport aux "
             "autres", bool(np.mean(growth_23[wrong0_23]) > np.mean(growth_23[~wrong0_23]) * (1 + 1e-9))),
            ("les poids avec lesquels chaque souche est entraînée somment à 1",
             all(np.isfinite(w).all() and abs(w.sum() - 1) < 1e-9 for w in weights_23)),
            ("les souches changent d'un tour à l'autre", len(cuts_23) > 1),
        ]
        all_ok_23, failed_23 = True, None
        for k, (what, ok) in enumerate(diagnostics_23, start=1):
            if not all_ok_23:
                print(f"⏳ Ex 14.23 : point {k} ({what}) : corrige d'abord le point {failed_23}.")
                continue
            verdict("14.23", ok, f"point {k} : {what}.", f"point {k} : attendu : {what}.")
            if not ok:
                all_ok_23, failed_23 = False, k
        with np.errstate(all="ignore"):
            accuracy_23 = float(np.mean(predict_23(stumps_23, alphas_23, X_23) == Y_23))
        print(f"training accuracy of the 40 rounds: {accuracy_23:.3f} · of the first stump alone: "
              f"{float(np.mean(stumps_23[0].predict(X_23) == Y_23)):.3f}")
        if not all_ok_23:
            print(f"⏳ Ex 14.23a et 14.23b : corrige d'abord le point {failed_23}.")
        else:                                            # a) and b) only once the three points pass
            wb.check("14.23a", accuracy_23, computed=True)
            wb.check("14.23b", float(alphas_23[1]), computed=True)
            oracle_23 = AdaBoostClassifier(n_estimators=40, random_state=0).fit(X_23, Y_23)
            verdict("14.23", np.array_equal(predict_23(stumps_23, alphas_23, X_23), oracle_23.predict(X_23))
                    and np.allclose(alphas_23, oracle_23.estimator_weights_),
                    "mêmes poids des souches et mêmes prédictions que l'AdaBoostClassifier de scikit-learn.",
                    "tes poids de souches ou tes prédictions diffèrent de ceux de l'AdaBoostClassifier de scikit-learn.")

### Ex 14.24 — Coder AdaBoost (SAMME) avec des souches pondérées 🔨 ★★★ ⏱️ 75 min
**Objectif :** écrire AdaBoost multi-classe sur tes souches, identique à celui de scikit-learn.  
**Prérequis :** ✏️ 14.6 · Ex 13.25 (`sample_weight`) · Ex 14.23 · fiche §14.7 (pseudo-code) · **Fil rouge :** synthétique · **mylearn :** `ensemble.py` · **Parcours :** C

> **mylearn** : dans `mon_travail/mylearn/ensemble.py`, mêmes règles qu'en 14.17. Enregistre, puis relance la cellule de vérification.

Écris la classe `AdaBoostClassifier`. **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `__init__` range ses quatre arguments tels quels ; le gabarit est `estimator`, ou `DecisionTreeClassifier(max_depth=1)` de **ton** `tree.py` si `estimator` vaut `None` ; il n'est jamais entraîné lui-même : chaque membre est un `clone`, qui reçoit, s'il a un `random_state`, un entier tiré d'un générateur `np.random.default_rng(random_state)` créé dans `fit` (comme le bagging) ;
- `fit` vérifie **d'abord** que le `fit` du gabarit a un paramètre `sample_weight`, avant tout entraînement ; puis les poids partent de $1/n$ ; à chaque tour : normaliser les poids (somme 1), entraîner le clone avec `sample_weight`, calculer `err` (la somme des poids des mal classés) ; si `err == 0`, garder le membre avec le poids 1,0 et l'erreur 0,0, et s'arrêter ; si `err >= 1 - 1/K`, rejeter le membre et s'arrêter (`ValueError` si c'était le tout premier : l'ensemble serait vide) ; sinon $\alpha = \eta\,(\ln\frac{1 - \text{err}}{\text{err}} + \ln(K - 1))$, garder le membre et multiplier par $e^\alpha$ le poids des mal classés ;
- attributs : `classes_` (labels triés, au moins 2 : `ValueError` sinon), `estimators_`, `estimator_weights_` et `estimator_errors_` (des tableaux NumPy, un élément par membre gardé) ; `ValueError` aussi pour `n_estimators < 1`, `learning_rate <= 0`, `NaN` ou `len(X) != len(y)` ;
- `decision_function(X)` rend un tableau `(n, K)` : pour chaque classe, la somme des $\alpha_m$ des membres qui la prédisent, divisée par la somme de tous les $\alpha_m$ ; `predict(X)` la classe de plus grand vote (à égalité, la première de `classes_`) ; `staged_predict(X)` est une **fonction génératrice** qui donne les prédictions après 1, 2, … membres ; `score(X, y)` l'accuracy, un `float` Python ; `RuntimeError` avant `fit`.

La vérification essaie l'exemple de la docstring, compare ton AdaBoost à celui de scikit-learn sur trois classes (poids, erreurs, prédictions après chaque tour, avec deux réglages), contrôle `decision_function`, puis lance les tests.

Dans tes notes : pourquoi vérifier la signature avant tout entraînement ? À deux classes, que vaut $\ln(K - 1)$ ? À trois classes, quel $\alpha$ la formule sans ce terme donnerait-elle à un membre qui se trompe sur 55 % des poids ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")   # reload your saved file
with checked("14.24"):
    X_doc_24, y_doc_24 = np.arange(1.0, 8.0).reshape(-1, 1), np.array([0, 0, 0, 1, 1, 0, 1])
    doc_24 = fitted(mylearn.ensemble.AdaBoostClassifier(n_estimators=3), X_doc_24, y_doc_24)
    print("docstring example: errors", np.round(np.asarray(doc_24.estimator_errors_, dtype=float), 4).tolist(),
          "(expected [0.1429, 0.1667, 0.2]) · weights", np.round(np.asarray(doc_24.estimator_weights_, dtype=float), 4).tolist(),
          "(expected [1.7918, 1.6094, 1.3863])")
with checked("14.24"):
    X_24, y_24 = wb.synth.make_blobs(n=400, centers=3, std=4.0, seed=1424)   # three overlapping classes
    X_24 = X_24.astype(np.float32).astype(float)                              # continuous data, no ties
    X_train_24, y_train_24, X_test_24 = X_24[:300], y_24[:300], X_24[300:]
    for params in [{"n_estimators": 20}, {"n_estimators": 15, "learning_rate": 0.5}]:
        start_24 = time.perf_counter()
        mine_24 = fitted(mylearn.ensemble.AdaBoostClassifier(**params), X_train_24, y_train_24)
        seconds_24 = time.perf_counter() - start_24
        oracle_24 = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), random_state=0, **params).fit(X_train_24, y_train_24)
        k_24 = len(mine_24.estimators_)
        same_weights_24 = (k_24 == len(oracle_24.estimators_)
                           and np.allclose(mine_24.estimator_weights_, oracle_24.estimator_weights_[:k_24])
                           and np.allclose(mine_24.estimator_errors_, oracle_24.estimator_errors_[:k_24]))
        same_stages_24 = all(np.array_equal(np.asarray(a), b) for a, b in
                             zip(mine_24.staged_predict(X_test_24), oracle_24.staged_predict(X_test_24)))
        verdict("14.24", same_weights_24 and same_stages_24,
                f"{params} : mêmes {k_24} poids, mêmes erreurs et mêmes prédictions après chaque tour que scikit-learn "
                f"(trois classes, en {fr(seconds_24, 2)} s).",
                f"{params} : scikit-learn garde {len(oracle_24.estimators_)} membres, de poids "
                f"{np.round(oracle_24.estimator_weights_[:3], 4).tolist()}… ; toi {k_24}, de poids "
                f"{np.round(np.asarray(mine_24.estimator_weights_[:3], dtype=float), 4).tolist()}… (ou d'autres "
                "prédictions) ; les tests ci-dessous disent où.")
    votes_24 = np.asarray(mine_24.decision_function(X_test_24), dtype=float)
    verdict("14.24", votes_24.shape == (100, 3) and np.allclose(votes_24.sum(axis=1), 1),
            "decision_function rend une ligne de 3 votes normalisés par exemple.",
            (f"decision_function : les lignes doivent sommer à 1 (divise par la somme des α), et non à "
             f"{np.round(votes_24.sum(axis=1)[:3], 4).tolist()}…" if votes_24.shape == (100, 3) else
             f"decision_function doit rendre un tableau (100, 3), une colonne par classe, pas {votes_24.shape}."))
if written(lambda: mylearn.ensemble.AdaBoostClassifier(n_estimators=1).fit(np.array([[0.0], [1.0]]), np.array([0, 1]))):
    run_mylearn_tests("ensemble", keyword="test_ada_boost_classifier_")
else:
    print("⏳ Ex 14.24 : les tests attendent le fit de ton AdaBoostClassifier.")

### Ex 14.25 — Coder un gradient boosting de régression 🔨 ★★★ ⏱️ 60 min
**Objectif :** écrire le gradient boosting de la fiche sur tes arbres de régression, identique à celui de scikit-learn.  
**Prérequis :** ✏️ 14.7 · Ex 13.26 · ch. 9 (MSE, R²) · fiche, section sur le gradient boosting (pseudo-code) · **Fil rouge :** California · **mylearn :** `ensemble.py` et `tree.py` · **Parcours :** M, C

> **mylearn** : dans `mon_travail/mylearn/ensemble.py` (et `tree.py`), mêmes règles qu'en 14.17. Enregistre, puis relance la cellule de vérification.

**D'abord, finis ton arbre de régression** (`tree.py`). Le 13.26 l'a laissé sans les options des ensembles : comme ton `DecisionTreeClassifier` au 13.25, ton `DecisionTreeRegressor` accepte `sample_weight` (un poids 2 équivaut à un exemple répété), `max_features` et `splitter="random"` (avec un générateur créé dans `fit` à partir de `random_state`), et calcule `feature_importances_` (la baisse de variance pondérée, normalisée). Ce sont les options d'une forêt de régression, comme celle de scikit-learn du 14.15.

**Puis écris la classe `GradientBoostingRegressor`** (`ensemble.py`). **Contrat** (la docstring, plus ces précisions que vérifient les tests) :
- `__init__` range ses six arguments tels quels ; `fit(X, y)` : `init_ = float(np.mean(y))` et $F = $ `init_` pour tous les exemples ; à chaque étape, les lignes de l'étape (toutes, ou, si `subsample < 1`, `rng.choice(n, size=max(1, int(subsample * n)), replace=False)`, avec **un** générateur `rng = np.random.default_rng(random_state)` créé dans `fit` : les tests comparent ces tirages étape après étape), un `clone` d'un `DecisionTreeRegressor(max_depth=..., min_samples_leaf=...)` de **ton** `tree.py` ajusté aux résidus $y - F$ de ces lignes, puis $F \leftarrow F + \eta \times$ `tree.predict(X)` pour **tous** les exemples ;
- `train_score_[m]` est la MSE après l'étape $m$ sur les lignes de l'étape (toutes si `subsample=1.0`) ; `estimators_` est la liste des arbres ; `ValueError` pour `n_estimators < 1`, `learning_rate <= 0`, `subsample` hors de $]0, 1]$, `NaN` ou `len(X) != len(y)` ;
- `predict(X)` rend `init_` plus $\eta$ fois la somme des prédictions des arbres, un tableau `float64` ; `staged_predict(X)` est une **fonction génératrice** qui donne les prédictions après 1, 2, … arbres ; `score(X, y)` rend le $R^2$, un `float` Python ; `RuntimeError` avant `fit`.

La vérification contrôle d'abord ton arbre de régression (poids, `max_features`, importances), essaie l'exemple de la docstring, compare ton boosting à celui de scikit-learn sur 400 districts (prédictions après chaque arbre, `init_`, `train_score_`), vérifie `subsample`, puis lance les tests du boosting et ceux de ton arbre de régression qui étaient réservés à ce chapitre (`test_decision_tree_regressor_ensemble_` du ch. 13).

Dans tes notes : pourquoi `predict` peut-il se réduire à parcourir `staged_predict` ? Que mesure `train_score_` avec `subsample=0.5`, et pourquoi n'est-ce plus la MSE d'entraînement ?

In [ ]:
mylearn = wb.load_mylearn("learner", missing_ok=True, fallback="ref", chapter="14")   # reload your saved file
with checked("14.25"):
    X_w_25, y_w_25 = wb.synth.make_moons(n=60, noise=0.3, seed=1425)
    X_w_25 = X_w_25.astype(np.float32).astype(float)
    t_w_25 = y_w_25 + X_w_25[:, 0]                                            # a real target
    weighted_25 = fitted(mylearn.tree.DecisionTreeRegressor(max_depth=4), X_w_25, t_w_25, sample_weight=np.full(60, 2.0))
    doubled_25 = fitted(mylearn.tree.DecisionTreeRegressor(max_depth=4), np.vstack([X_w_25, X_w_25]), np.concatenate([t_w_25, t_w_25]))
    first_25 = mylearn.tree.DecisionTreeRegressor(max_features=1, random_state=0)
    a_25 = fitted(first_25, X_w_25, t_w_25).predict(X_w_25)
    b_25 = fitted(first_25, X_w_25, t_w_25).predict(X_w_25)
    oracle_tree_25 = DecisionTreeRegressor(max_depth=4, random_state=0).fit(X_w_25, t_w_25)
    mine_tree_25 = fitted(mylearn.tree.DecisionTreeRegressor(max_depth=4), X_w_25, t_w_25)
    verdict("14.25", np.allclose(weighted_25.predict(X_w_25), doubled_25.predict(X_w_25)) and np.array_equal(a_25, b_25)
            and np.allclose(mine_tree_25.feature_importances_, oracle_tree_25.feature_importances_),
            "ton DecisionTreeRegressor a les options du 13.25 : poids, max_features reproductible, importances.",
            "ton DecisionTreeRegressor doit accepter sample_weight (un poids 2 = un exemple répété), rendre "
            "max_features reproductible avec random_state et calculer feature_importances_ comme scikit-learn ; les "
            "tests du ch. 13 lancés plus bas disent où.")
with checked("14.25"):
    doc_25 = fitted(mylearn.ensemble.GradientBoostingRegressor(n_estimators=2, learning_rate=0.5, max_depth=1),
                    np.array([[1.0], [2.0], [3.0], [4.0]]), np.array([1.0, 1.2, 3.0, 3.4]))
    staged_doc_25 = [np.round(np.asarray(p, dtype=float), 4).tolist() for p in doc_25.staged_predict(np.array([[0.0], [5.0]]))]
    print("docstring example: init_", round(float(doc_25.init_), 4), "(expected 2.15) · staged", staged_doc_25,
          "(expected [[1.625, 2.675], [1.3625, 2.9375]]) · train_score_",
          np.round(np.asarray(doc_25.train_score_, dtype=float), 4).tolist(), "(expected [0.3006, 0.0939])")
with checked("14.25"):
    rows_25 = np.random.default_rng(1425).choice(len(X_CAL_TRAIN), 1400, replace=False)
    X_25 = X_CAL_TRAIN[rows_25].astype(np.float32).astype(float)               # continuous data, no ties
    y_25 = Y_CAL_TRAIN[rows_25]
    X_train_25, y_train_25, X_test_25, y_test_25 = X_25[:400], y_25[:400], X_25[400:], y_25[400:]
    params_25 = {"n_estimators": 20, "learning_rate": 0.3, "max_depth": 2, "min_samples_leaf": 10}   # no tiny leaves: no ties
    start_25 = time.perf_counter()
    mine_25 = fitted(mylearn.ensemble.GradientBoostingRegressor(**params_25), X_train_25, y_train_25)
    seconds_25 = time.perf_counter() - start_25
    oracle_25 = GradientBoostingRegressor(criterion="squared_error", random_state=0, **params_25).fit(X_train_25, y_train_25)
    stages_25 = list(mine_25.staged_predict(X_test_25))
    same_25 = (len(stages_25) == 20 and abs(float(mine_25.init_) - float(np.mean(y_train_25))) < 1e-12
               and all(np.allclose(a, b) for a, b in zip(stages_25, oracle_25.staged_predict(X_test_25)))
               and np.allclose(mine_25.train_score_, oracle_25.train_score_))
    verdict("14.25", same_25,
            f"mêmes prédictions après chacun des 20 arbres et même train_score_ que scikit-learn (en {fr(seconds_25, 2)} s) ; "
            f"RMSE de test {fr(rmse(y_test_25, stages_25[-1]))}.",
            "tes prédictions après chaque arbre, ton init_ ou ton train_score_ diffèrent de ceux de scikit-learn ; les "
            "tests ci-dessous disent où.")
    sub_a_25 = fitted(mylearn.ensemble.GradientBoostingRegressor(subsample=0.5, random_state=3, **params_25), X_train_25, y_train_25)
    sub_b_25 = fitted(mylearn.ensemble.GradientBoostingRegressor(subsample=0.5, random_state=3, **params_25), X_train_25, y_train_25)
    verdict("14.25", np.array_equal(sub_a_25.predict(X_test_25), sub_b_25.predict(X_test_25))
            and all(tree.root_.n_samples == 200 for tree in sub_a_25.estimators_),
            "subsample=0.5 : chaque arbre apprend sur 200 exemples tirés, et le même random_state redonne le même modèle.",
            "subsample=0.5 : chaque arbre doit apprendre sur max(1, int(0.5 × 400)) = 200 exemples tirés sans remise, "
            "avec un générateur créé dans fit (le même random_state redonne le même modèle).")
if written(lambda: mylearn.ensemble.GradientBoostingRegressor(n_estimators=1).fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))):
    run_mylearn_tests("ensemble", keyword="test_gradient_boosting_regressor_")
else:
    print("⏳ Ex 14.25 : les tests attendent le fit de ton GradientBoostingRegressor.")
if written(lambda: mylearn.tree.DecisionTreeRegressor().fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))):
    run_mylearn_tests("tree", keyword="test_decision_tree_regressor_ensemble_")
else:
    print("⏳ Ex 14.25 : les tests du ch. 13 attendent ton DecisionTreeRegressor.")

## Partie D · Le gradient boosting en pratique

Les trois bibliothèques de gradient boosting à histogrammes avec early stopping (14.26), un benchmark reproductible (14.27), puis le défi sur California (14.28). La cellule suivante importe XGBoost et LightGBM.

In [ ]:
# The gradient boosting libraries: XGBoost and LightGBM (installed on Colab and by requirements.txt)
import platform
import sklearn

warnings.filterwarnings("ignore", message="X does not have valid feature names")   # LightGBM names its columns itself

try:
    import lightgbm
    import xgboost
except ImportError as exc:                                # wb.ensure installs a missing package at its pinned version
    print(f"⏳ {exc.name} absent : wb.ensure(\"{exc.name}\") l'installe à la version du workbook.")
print("xgboost", getattr(globals().get("xgboost"), "__version__", "absent"), "· lightgbm",
      getattr(globals().get("lightgbm"), "__version__", "absent"))

### Ex 14.26 — HistGradientBoosting, XGBoost et LightGBM sur California 📦 ★★★ ⏱️ 45 min
**Objectif :** entraîner trois gradient boostings à histogrammes avec early stopping, et lire leurs nombres d'arbres.  
**Prérequis :** Ex 14.20 · ch. 8 (jeu de validation) · ch. 9 (early stopping) · fiche, section sur le gradient boosting · **Fil rouge :** California · **Parcours :** R, C

Les 16 512 districts d'entraînement, coupés en 13 209 pour entraîner (`X_FIT_26`) et 3 303 pour valider (`X_VAL_26`) ; le test (`X_CAL_TEST`) ne sert qu'à la mesure finale. Les trois modèles ont le même learning rate, 0,1, et au plus 1 000 arbres ; l'early stopping s'arrête après 20 arbres sans amélioration de la RMSE de validation.
- **HistGradientBoosting** (dans scikit-learn 1.6, son `fit` n'accepte pas de jeu de validation fourni par toi) : entraîne-le **sans** early stopping, calcule sa courbe de validation avec `staged_predict`, puis fais l'early stopping toi-même avec `best_iteration_26(curve, patience=20)`, que tu écris : parcourir la courbe depuis le début, s'arrêter dès que `patience` arbres de suite n'ont pas battu (strictement) la meilleure valeur, et rendre le nombre d'arbres (compté à partir de 1) de cette meilleure valeur.
- **XGBoost** et **LightGBM** font l'early stopping eux-mêmes sur `eval_set` ; leurs `predict` utilisent ensuite le meilleur nombre d'arbres. Attention : `best_iteration` de XGBoost compte à partir de 0, `best_iteration_` de LightGBM à partir de 1.

Calcule les variables de la cellule, dont :
a) `hgb_trees_26` : le nombre d'arbres que garde ton early stopping ;
b) `hgb_rmse_26` : la RMSE de test de HistGradientBoosting réduit à ces arbres (`staged_predict` sur le test) ;
c) `xgb_trees_26` : le nombre d'arbres que garde XGBoost ; d) `xgb_rmse_26` : sa RMSE de test ;
e) `lgbm_trees_26` : le nombre d'arbres que garde LightGBM ; f) `lgbm_rmse_26` : sa RMSE de test ;
et `seconds_26`, les durées d'entraînement.

Les RMSE de XGBoost et de LightGBM se vérifient à 2 décimales seulement : leur calcul peut changer à la marge d'une machine à l'autre. Dans tes notes : classe les trois en RMSE et en temps. Pourquoi les trois ne gardent-ils pas le même nombre d'arbres, avec le même learning rate ? Pourquoi l'early stopping se règle-t-il sur la validation et pas sur le test ? À combien de la barre de 0,45 du défi (🏆 14.28) es-tu ?

In [ ]:
X_FIT_26, X_VAL_26, Y_FIT_26, Y_VAL_26 = train_test_split(X_CAL_TRAIN, Y_CAL_TRAIN, test_size=0.2, random_state=1426)
print(len(X_FIT_26), "districts to fit,", len(X_VAL_26), "to validate (early stopping),", len(X_CAL_TEST), "to test")

In [ ]:
def best_iteration_26(curve, patience=20):
    """`curve` holds the validation RMSE after 1, 2, 3... trees. Scan it from the start and stop as soon as `patience`
    trees in a row have not improved on the best value so far (improve = strictly lower). Return the number of trees
    (counted from 1) at that best value."""
    raise NotImplementedError("best_iteration_26")


hgb_26 = ...          # HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.1, early_stopping=False,
                      #   random_state=0) fitted on X_FIT_26, Y_FIT_26
hgb_curve_26 = ...    # the validation RMSE of hgb_26 after each tree (staged_predict on X_VAL_26)
hgb_trees_26 = ...    # a) best_iteration_26(hgb_curve_26)
hgb_rmse_26 = ...     # b) the test RMSE (X_CAL_TEST) of hgb_26 when it keeps its first hgb_trees_26 trees
xgb_26 = ...          # xgboost.XGBRegressor(n_estimators=1000, learning_rate=0.1, early_stopping_rounds=20,
                      #   random_state=0), fitted on X_FIT_26, Y_FIT_26 with eval_set=[(X_VAL_26, Y_VAL_26)], verbose=False
xgb_trees_26 = ...    # c) the number of trees XGBoost keeps (best_iteration counts from 0)
xgb_rmse_26 = ...     # d) the test RMSE of xgb_26 (its predict uses the best iteration)
lgbm_26 = ...         # lightgbm.LGBMRegressor(n_estimators=1000, learning_rate=0.1, random_state=0, verbose=-1), fitted on
                      #   X_FIT_26, Y_FIT_26 with eval_set=[(X_VAL_26, Y_VAL_26)],
                      #   callbacks=[lightgbm.early_stopping(20, verbose=False)]
lgbm_trees_26 = ...   # e) the number of trees LightGBM keeps (best_iteration_ counts from 1)
lgbm_rmse_26 = ...    # f) the test RMSE of lgbm_26
seconds_26 = ...      # dict {"hgb": ..., "xgb": ..., "lgbm": ...}: the fit time of each model, in seconds

In [ ]:
wb.check("14.26a", hgb_trees_26)
wb.check("14.26b", hgb_rmse_26)
with wb.attempt("14.26"):
    if not filled(xgb_26, xgb_trees_26, xgb_rmse_26, lgbm_26, lgbm_trees_26, lgbm_rmse_26):
        raise NotImplementedError("xgb_26, xgb_trees_26, xgb_rmse_26, lgbm_26, lgbm_trees_26, lgbm_rmse_26")
    params_ok_26 = (xgb_26.get_params()["learning_rate"] == 0.1 and xgb_26.get_params()["n_estimators"] == 1000
                    and xgb_26.get_params()["early_stopping_rounds"] == 20 and lgbm_26.get_params()["learning_rate"] == 0.1
                    and lgbm_26.get_params()["n_estimators"] == 1000 and getattr(lgbm_26, "best_iteration_", 0) >= 1)
    verdict("14.26", params_ok_26,
            "les deux modèles ont les réglages de l'énoncé (learning rate 0,1, au plus 1 000 arbres) et un early stopping.",
            "relis les réglages : learning_rate=0.1 et n_estimators=1000 pour les deux, early_stopping_rounds=20 pour "
            "XGBoost, et pour LightGBM eval_set avec callbacks=[lightgbm.early_stopping(20, verbose=False)] "
            "(sans early stopping, best_iteration_ vaut 0).")
    on_test_26 = (abs(float(xgb_rmse_26) - rmse(Y_CAL_TEST, xgb_26.predict(X_CAL_TEST))) < 1e-9
                  and abs(float(lgbm_rmse_26) - rmse(Y_CAL_TEST, lgbm_26.predict(X_CAL_TEST))) < 1e-9)
    verdict("14.26", on_test_26, "d) et f) sont les RMSE de test des deux modèles.",
            "d) et f) sont les RMSE sur le test (X_CAL_TEST, Y_CAL_TEST) des modèles entraînés, pas sur la validation.")
    if not (params_ok_26 and on_test_26):
        raise NotImplementedError("c) à f) : corrige d'abord les points signalés ci-dessus")
    verdict("14.26", xgb_trees_26 == xgb_26.best_iteration + 1,
            f"c) XGBoost garde {xgb_trees_26} arbres (best_iteration = {xgb_26.best_iteration}, compté à partir de 0).",
            f"c) XGBoost garde best_iteration + 1 arbres, soit {xgb_26.best_iteration + 1} : best_iteration compte "
            "à partir de 0.")
    wb.check("14.26d", xgb_rmse_26)
    verdict("14.26", lgbm_trees_26 == lgbm_26.best_iteration_,
            f"e) LightGBM garde {lgbm_trees_26} arbres (best_iteration_, compté à partir de 1).",
            f"e) LightGBM garde best_iteration_ arbres, soit {lgbm_26.best_iteration_} (compté à partir de 1).")
    wb.check("14.26f", lgbm_rmse_26)
    if filled(seconds_26):
        print("fit times (s):", {name: round(float(seconds), 2) for name, seconds in dict(seconds_26).items()})

### Ex 14.27 — Un benchmark reproductible : graines, versions et tableau de résultats 🛠️ ★★ ⏱️ 25 min
**Objectif :** écrire un petit banc d'essai qu'un collègue pourra relancer : graines, plusieurs exécutions, versions notées.  
**Prérequis :** Ex 14.26 · ch. 2 (moyenne, écart-type) · fiche, encadré « fixer les graines d'un ensemble » · **Fil rouge :** California · **Parcours :** C

Un tableau de résultats ne vaut que si on peut le refaire. Écris quatre fonctions :
- `versions_27()` : un dictionnaire des versions de `"python"` (`platform.python_version()`), `"numpy"`, `"scikit-learn"`, `"xgboost"` et `"lightgbm"` (leur `__version__`), plus `"cores"`, le nombre de cœurs (`os.cpu_count()`) ;
- `benchmark_27(models, X_train, y_train, X_test, y_test, seeds)` : `models` associe un nom à une fonction `graine → modèle neuf` (`MODELS_27` : une régression ridge, une forêt de 50 arbres, un HistGradientBoosting) ; pour chaque modèle et chaque graine de `seeds`, construire le modèle, l'entraîner, mesurer la durée de `fit` (`time.perf_counter`) et la RMSE de test ; rendre un `DataFrame` d'une ligne par couple (modèle, graine), aux colonnes `"model"`, `"seed"`, `"rmse"` et `"fit_seconds"` ;
- `summary_27(results)` : une ligne par modèle, indexée par son nom et triée par RMSE moyenne croissante, aux colonnes `"rmse_mean"`, `"rmse_std"` (écart-type avec `ddof=1`) et `"fit_seconds_mean"` ;
- `markdown_27(summary, versions)` : le texte d'un tableau Markdown du résumé (une ligne d'en-tête, une ligne `|---|`, une ligne par modèle, la RMSE avec 4 décimales et les secondes avec 2), suivi d'une ligne vide et d'une ligne qui liste les versions (`DataFrame.to_markdown` demande le paquet `tabulate`, absent : écris les lignes toi-même).

La vérification contrôle les versions, la forme du tableau, la reproductibilité (le benchmark relancé redonne les mêmes RMSE), l'effet des graines, le résumé et le texte, qu'elle affiche. Cet exercice se corrige avec `05_solutions.md`.

Dans tes notes : qu'est-ce qui, dans ce tableau, changerait sur une autre machine, et qu'est-ce qui ne changerait pas ? Pourquoi trois graines plutôt qu'une ? Où rangerais-tu ce tableau dans un dépôt, et avec quel commit ?

In [ ]:
X_TRAIN_27, Y_TRAIN_27 = X_CAL_TRAIN[:5000], Y_CAL_TRAIN[:5000]
MODELS_27 = {"ridge": lambda seed: make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
             "forest": lambda seed: RandomForestRegressor(n_estimators=50, random_state=seed, n_jobs=-1),
             "hgb": lambda seed: HistGradientBoostingRegressor(random_state=seed)}
SEEDS_27 = [0, 1, 2]

In [ ]:
def versions_27():
    """Return a dict {name: version} for "python", "numpy", "scikit-learn", "xgboost" and "lightgbm", plus "cores":
    the number of CPU cores (os.cpu_count())."""
    raise NotImplementedError("versions_27")


def benchmark_27(models, X_train, y_train, X_test, y_test, seeds):
    """`models` maps a name to a function seed -> NEW unfitted regressor. For every name and every seed: build the
    model, fit it, measure the fit time (time.perf_counter) and the test RMSE. Return a pandas DataFrame with one row
    per (model, seed) and the columns "model", "seed", "rmse", "fit_seconds"."""
    raise NotImplementedError("benchmark_27")


def summary_27(results):
    """One row per model, sorted by increasing mean RMSE: the columns "rmse_mean", "rmse_std" (ddof=1) and
    "fit_seconds_mean" (a DataFrame indexed by the model name)."""
    raise NotImplementedError("summary_27")


def markdown_27(summary, versions):
    """A Markdown table of `summary` (a header line, a |---| line, one line per model, the numbers with 4 decimals
    for the RMSE and 2 for the seconds), followed by an empty line and one line that lists the versions."""
    raise NotImplementedError("markdown_27")

In [ ]:
with checked("14.27"):
    versions_found_27 = versions_27()
    if returned("14.27", "versions_27", versions_found_27):
        expected_27 = {"python": platform.python_version(), "numpy": np.__version__, "scikit-learn": sklearn.__version__,
                       "xgboost": xgboost.__version__, "lightgbm": lightgbm.__version__, "cores": os.cpu_count()}
        verdict("14.27", {k: str(v) for k, v in dict(versions_found_27).items()} == {k: str(v) for k, v in expected_27.items()},
                "versions_27 rend les six informations attendues.",
                f"versions_27 doit rendre exactement les clés {list(expected_27)} et les versions installées.")
with checked("14.27"):
    results_27 = benchmark_27(MODELS_27, X_TRAIN_27, Y_TRAIN_27, X_CAL_TEST, Y_CAL_TEST, SEEDS_27)
    if returned("14.27", "benchmark_27", results_27):
        shape_ok_27 = (isinstance(results_27, pd.DataFrame) and list(results_27.columns) == ["model", "seed", "rmse", "fit_seconds"]
                       and len(results_27) == 9)
        verdict("14.27", shape_ok_27, "benchmark_27 rend 9 lignes (3 modèles × 3 graines) et les 4 colonnes attendues.",
                "benchmark_27 doit rendre un DataFrame de 9 lignes, aux colonnes model, seed, rmse, fit_seconds.")
        if shape_ok_27:
            again_27 = benchmark_27(MODELS_27, X_TRAIN_27, Y_TRAIN_27, X_CAL_TEST, Y_CAL_TEST, SEEDS_27)
            verdict("14.27", np.allclose(results_27["rmse"], again_27["rmse"], rtol=0, atol=1e-9),
                    "relancé, le benchmark redonne les mêmes RMSE : il est reproductible.",
                    "relancé, le benchmark donne d'autres RMSE : chaque modèle reçoit-il bien sa graine, et un modèle "
                    "neuf à chaque appel ?")
            spread_27 = results_27.groupby("model")["rmse"].std(ddof=1)
            verdict("14.27", spread_27["ridge"] < 1e-12 and spread_27["forest"] > 1e-6,
                    "la régression ridge ne dépend pas de la graine, la forêt si.",
                    "attendu : un écart nul entre graines pour ridge (aucun hasard), non nul pour la forêt.")
            table_27 = summary_27(results_27)
            text_27 = markdown_27(table_27, versions_found_27)
            lines_27 = str(text_27).splitlines()
            verdict("14.27", list(table_27.columns) == ["rmse_mean", "rmse_std", "fit_seconds_mean"]
                    and list(table_27["rmse_mean"]) == sorted(table_27["rmse_mean"])
                    and len(lines_27) >= 6 and lines_27[0].startswith("|") and lines_27[1].replace(" ", "").startswith("|---")
                    and "scikit-learn" in lines_27[-1],
                    "summary_27 trie les modèles par RMSE moyenne, et markdown_27 rend un tableau suivi des versions.",
                    "attendu : summary_27 aux colonnes rmse_mean, rmse_std, fit_seconds_mean, trié par rmse_mean ; "
                    "markdown_27 : une ligne d'en-tête, une ligne |---|, une ligne par modèle, puis les versions.")
            print(text_27)

### Ex 14.28 — Défi : RMSE ≤ 0,45 sur California 🏆 ★★★★ ⏱️ 100 min
**Objectif :** construire et régler un modèle tabulaire qui passe sous 0,45 de RMSE, sur le test et sur d'autres découpages.  
**Prérequis :** Ex 14.26 · ch. 8 (validation croisée) · ch. 12 (préparation des features) · fiche, section sur le gradient boosting · **Fil rouge :** California · **Parcours :** C

**Objectif** : une RMSE de test d'au plus **0,45** sur le découpage du défi (20 % de test, `random_state=42`, celui des parties B à D), **et** au plus 0,45 en moyenne sur trois autres découpages (`random_state` = 1, 2 et 3), ta méthode étant réentraînée sur chacun. Palier 🌟 : les deux sous 0,43. Écris `build_model_28()`, qui rend un modèle **neuf**, non entraîné (scikit-learn, XGBoost ou LightGBM, éventuellement un pipeline), qui reçoit les 8 features brutes de California (forme `(n, 8)`, dans l'ordre de `CAL_FEATURES`) et prédit `MedHouseVal`.

Règles : règle ta méthode par validation croisée (ou early stopping) sur les données d'entraînement, dans une cellule à toi ; les quatre jeux de test ne servent qu'à la mesure finale, révélée **une fois par session** (relancer la cellule redonne le même résultat ; redémarre le noyau pour une nouvelle tentative). Pour te situer : `HistGradientBoostingRegressor(random_state=0)`, avec ses réglages par défaut, fait 0,462 sur le test du défi et 0,466 en moyenne sur les trois autres. La vérification refuse un modèle déjà entraîné, ou le même objet rendu deux fois, signale une RMSE trop belle pour être honnête (sous 0,35) et réentraîne ta méthode une fois sur des cibles mélangées au hasard, sur lesquelles elle ne doit rien apprendre (sinon, ton modèle voit-il les lignes de test ?) ; le reste est une règle d'honneur : ne règle rien sur les découpages de la vérification, et ne fais pas oublier à la cellule ce qu'elle a révélé. Ce défi se corrige avec `05_solutions.md`.

Dans tes notes : les modèles et les réglages essayés, avec leurs scores de validation croisée ; ce qui a le plus aidé (les hyperparamètres, le learning rate et l'early stopping, de nouvelles features) ; le temps d'entraînement de ton modèle.

In [ ]:
def build_model_28():
    """Return a NEW, unfitted regressor (scikit-learn, XGBoost or LightGBM, possibly a pipeline) that receives the 8 raw
    features of California (shape (n, 8), in the order of CAL_FEATURES) and predicts MedHouseVal."""
    raise NotImplementedError("build_model_28")

In [ ]:
from sklearn.exceptions import NotFittedError
from sklearn.utils.validation import check_is_fitted


def is_fitted_28(model):
    """True when check_is_fitted says the model was already fitted."""
    try:
        check_is_fitted(model)
    except NotFittedError:
        return False
    except Exception:                                   # not a scikit-learn-like model: the fit below will say
        return False
    return True


with checked("14.28"):
    if globals().get("REVEALED_28") is None:
        scores_28 = {}
        first_28, second_28 = build_model_28(), build_model_28()
        accepted_28 = returned("14.28", "build_model_28", first_28)
        if accepted_28 and (isinstance(first_28, type) or is_fitted_28(first_28)):
            print("❌ Ex 14.28 : build_model_28 doit rendre un modèle (une instance) neuf, non entraîné.")
            accepted_28 = False
        elif accepted_28 and first_28 is second_28:
            print("❌ Ex 14.28 : build_model_28 rend deux fois le même objet : il doit construire un modèle NEUF à chaque appel.")
            accepted_28 = False
        for split_seed in ([42, 1, 2, 3] if accepted_28 else []):   # the challenge, then three other splits
            X_tr, X_te, y_tr, y_te = train_test_split(X_CAL, Y_CAL, test_size=0.2, random_state=split_seed)
            model_28 = build_model_28()
            start_28 = time.perf_counter()
            model_28.fit(X_tr, y_tr)
            scores_28[split_seed] = (rmse(y_te, model_28.predict(X_te)), time.perf_counter() - start_28)
        if len(scores_28) == 4:                         # a control: the same method on shuffled targets must learn nothing
            X_tr, X_te, y_tr, y_te = train_test_split(X_CAL, Y_CAL, test_size=0.2, random_state=42)
            control_28 = build_model_28()
            control_28.fit(X_tr, np.random.default_rng(1428).permutation(y_tr))
            scores_28["shuffled"] = (rmse(y_te, control_28.predict(X_te)), 0.0)
            REVEALED_28 = scores_28                     # revealed once per session: kept if the cell runs again
    if globals().get("REVEALED_28") is not None:
        main_28 = REVEALED_28[42][0]
        others_28 = float(np.mean([REVEALED_28[s][0] for s in (1, 2, 3)]))
        print(f"test RMSE, split random_state=42: {main_28:.4f} (fit in {REVEALED_28[42][1]:.1f} s) · mean over the "
              f"splits 1, 2, 3: {others_28:.4f} ({', '.join(f'{REVEALED_28[s][0]:.4f}' for s in (1, 2, 3))})")
        shuffled_28 = REVEALED_28["shuffled"][0]
        if min(main_28, others_28) < 0.35 or shuffled_28 < 0.9:
            print(f"⚠️ Ex 14.28 : le défi n'est pas validé : {'une RMSE sous 0,35 est trop belle pour être honnête' if min(main_28, others_28) < 0.35 else ''}"
                  f"{' ; ' if min(main_28, others_28) < 0.35 and shuffled_28 < 0.9 else ''}"
                  f"{f'entraîné sur des cibles mélangées au hasard, ton modèle fait encore {fr(shuffled_28)} (un modèle honnête ne fait pas mieux que 1,15 environ)' if shuffled_28 < 0.9 else ''}"
                  " : ton modèle voit-il les lignes de test (des variables globales, un fit qui ignore ses arguments) ?")
        else:
            verdict("14.28", main_28 <= 0.45 and others_28 <= 0.45,
                    f"défi réussi : {fr(main_28, 4)} sur le test du défi et {fr(others_28, 4)} en moyenne sur trois "
                    "autres découpages." + (" 🌟 Palier atteint : les deux sous 0,43." if max(main_28, others_28) <= 0.43 else ""),
                    f"{fr(main_28, 4)} sur le test du défi, {fr(others_28, 4)} en moyenne ailleurs : les deux doivent "
                    "être au plus 0,45 ; règle ta méthode par validation croisée sur l'entraînement, jamais sur ces "
                    "tests (redémarre le noyau pour une nouvelle tentative).")

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu dire quand un ensemble bat ses membres, et pourquoi il ne corrige pas un biais commun ?
2. Sais-tu dérouler à la main un vote pondéré, un tour d'AdaBoost et une étape de gradient boosting ?
3. Sais-tu choisir entre une forêt et un gradient boosting, et régler le nombre d'arbres et le learning rate ?

**Pour aller plus loin** : les articles de Breiman (1996, 2001), de Freund et Schapire (1997) et de Friedman (2001), et le guide utilisateur « Ensembles » de scikit-learn, cités dans la fiche. La suite : le ch. 15 (scikit-learn), qui mettra ces ensembles dans des pipelines et des recherches d'hyperparamètres.